In [ ]:
# ============================================================
# CELL 1 — Install ALL packages (run after every restart)
# ============================================================

!pip install sentence-transformers InstructorEmbedding textstat xgboost -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)

print("✅ All packages installed!")

In [ ]:
# ============================================================
# STEP 1: Install Libraries & Load Data
# ============================================================

!pip install sentence-transformers textstat -q

import pandas as pd
import numpy as np
import nltk
import warnings
from sklearn.metrics import cohen_kappa_score
warnings.filterwarnings('ignore')

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')

# --- Find your file path first ---
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

**MALAYALAM DATASET**

In [1]:
# ============================================================
# MALAYALAM AES — COMPLETE PIPELINE
# Dataset: 3000 essays, 150 prompts, 20 essays/prompt
# Score range: 0-10 all prompts
# KEY ADVANTAGE: Has ref_answer → semantic similarity!
# Model: multilingual-e5-large-instruct
# ============================================================

# ── STEP 1: Install ────────────────────────────────────────
# !pip install sentence-transformers -q

import os, gc, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR = '/kaggle/working/'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── STEP 2: Load Dataset ───────────────────────────────────
print("Loading datasets...")
df = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/mal_dataset.csv'
)
q_df = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/questions_150.csv'
)

# Merge prompt text into main df
df = df.merge(
    q_df[['prompt_id','prompt','max_marks']],
    on='prompt_id', how='left'
)

print(f"✅ Essays loaded: {df.shape}")
print(f"   Prompts: {df['prompt_id'].nunique()}")
print(f"   Essays per prompt: "
      f"{df.groupby('prompt_id').size().mean():.1f}")
print(f"   Score range: "
      f"{df['essay_score'].min()}-"
      f"{df['essay_score'].max()}")
print(f"   Has ref_answer: "
      f"{df['ref_answer'].notna().all()}")

# Sanity check
print("\nScore distribution:")
print(df['essay_score'].value_counts().sort_index())

Loading datasets...
✅ Essays loaded: (3000, 7)
   Prompts: 150
   Essays per prompt: 20.0
   Score range: 0-10
   Has ref_answer: True

Score distribution:
essay_score
0      66
1     279
2     345
3     424
4     406
5     385
6     270
7     250
8     187
9     107
10    281
Name: count, dtype: int64


In [2]:
# ============================================================
# STEP 3: Malayalam Shallow Features
# ============================================================

def get_malayalam_features(text, ref_text=None):
    """
    Malayalam-specific features.
    Malayalam Unicode range: U+0D00–U+0D7F
    """
    if not isinstance(text, str):
        text = ''

    # ── Word-level features ─────────────────────────────
    words = text.split()
    n_words = max(len(words), 1)

    # ── Sentence-level features ─────────────────────────
    # Malayalam sentence boundary: ., ?, !, ।
    import re
    sents = [
        s.strip() for s in
        re.split(r'[.?!।]', text)
        if s.strip()
    ]
    n_sents = max(len(sents), 1)

    # ── Malayalam script features ────────────────────────
    # Check how much of the essay is Malayalam Unicode
    mal_chars = sum(
        1 for c in text
        if '\u0D00' <= c <= '\u0D7F'
    )
    total_alpha = sum(1 for c in text if c.isalnum())
    mal_ratio = mal_chars / max(total_alpha, 1)

    # ── Lexical diversity ────────────────────────────────
    unique_words = set(words)
    ttr = len(unique_words) / n_words  # type-token ratio

    # ── Sentence statistics ──────────────────────────────
    sent_lens = [len(s.split()) for s in sents]
    avg_sent_len = n_words / n_sents
    sent_len_std = np.std(sent_lens) \
                   if len(sent_lens) > 1 else 0

    # ── Character-level features ─────────────────────────
    avg_word_len = np.mean([
        len(w) for w in words
    ]) if words else 0

    # ── Paragraph features ───────────────────────────────
    paras = [
        p.strip() for p in text.split('\n')
        if p.strip()
    ]
    n_paras = max(len(paras), 1)

    # ── Punctuation diversity ────────────────────────────
    puncts = set(
        c for c in text
        if not c.isalnum() and not c.isspace()
    )
    punct_diversity = len(puncts)

    return {
        'word_count'         : n_words,
        'sent_count'         : n_sents,
        'para_count'         : n_paras,
        'avg_sent_len'       : round(avg_sent_len, 3),
        'sent_len_std'       : round(sent_len_std, 3),
        'avg_word_len'       : round(avg_word_len, 3),
        'type_token_ratio'   : round(ttr, 3),
        'malayalam_ratio'    : round(mal_ratio, 3),
        'punct_diversity'    : punct_diversity,
        'char_count'         : len(text),
    }

print("Computing Malayalam shallow features...")
from tqdm import tqdm

sf_records = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    feats = get_malayalam_features(
        row['essay'], row['ref_answer']
    )
    sf_records.append(feats)

sf_df = pd.DataFrame(sf_records)
sf_df.to_csv(
    os.path.join(SAVE_DIR,
                 'mal_shallow_features.csv'),
    index=False
)
print(f"✅ Shallow features: {sf_df.shape}")
print(sf_df.describe().round(3))

Computing Malayalam shallow features...


100%|██████████| 3000/3000 [00:01<00:00, 2605.04it/s]

✅ Shallow features: (3000, 10)
       word_count  sent_count  para_count  avg_sent_len  sent_len_std  \
count    3000.000    3000.000    3000.000      3000.000      3000.000   
mean       63.017       7.541       1.005         8.707         2.794   
std        52.746       6.290       0.075         2.760         1.861   
min         3.000       1.000       1.000         2.000         0.000   
25%        27.000       3.000       1.000         6.728         1.564   
50%        48.000       6.000       1.000         8.297         2.608   
75%        81.250      10.000       1.000        10.167         3.785   
max       456.000      59.000       3.000        31.000        16.183   

       avg_word_len  type_token_ratio  malayalam_ratio  punct_diversity  \
count      3000.000          3000.000         3000.000         3000.000   
mean          9.123             0.890            1.631           13.233   
std           0.881             0.082            0.058            2.734   
min        

In [3]:
# ============================================================
# STEP 4: Multilingual E5 Embeddings for Malayalam
# KEY FEATURE: Essay vs Reference Answer similarity
# ============================================================

MAL_EMB_PATH   = os.path.join(
    SAVE_DIR, 'mal_embeddings.csv'
)
MAL_CKPT_DIR   = os.path.join(
    SAVE_DIR, 'mal_emb_checkpoints'
)
os.makedirs(MAL_CKPT_DIR, exist_ok=True)

if os.path.exists(MAL_EMB_PATH):
    print("✅ Embeddings exist!")
    emb_df = pd.read_csv(MAL_EMB_PATH)
    print("Shape:", emb_df.shape)

else:
    gc.collect()
    torch.cuda.empty_cache()

    texts      = df['essay'].astype(str).tolist()
    ref_texts  = df['ref_answer'].astype(str).tolist()
    prompt_ids = df['prompt_id'].tolist()
    n_total    = len(texts)
    CHUNK_SIZE = 500

    # Instruction for Malayalam essay scoring
    ESSAY_INSTR = (
        "Instruct: Represent this Malayalam student "
        "answer for evaluating content accuracy and "
        "completeness against a reference answer\n"
        "Query: "
    )
    REF_INSTR = (
        "Instruct: Represent this Malayalam reference "
        "answer as the gold standard for scoring\n"
        "Query: "
    )
    SENT_INSTR = (
        "Instruct: Represent this Malayalam sentence "
        "for measuring coherence\nQuery: "
    )

    # Find completed chunks
    done = sorted([
        int(f.replace('chunk_','').replace('.npz',''))
        for f in os.listdir(MAL_CKPT_DIR)
        if f.startswith('chunk_') and f.endswith('.npz')
    ])
    start_idx = (max(done)+1)*CHUNK_SIZE if done else 0

    print(f"Total: {n_total}, Starting: {start_idx}")
    print(f"GPU free: "
          f"{torch.cuda.mem_get_info()[0]/1024**3:.1f}GB")

    if start_idx < n_total:
        MODEL = 'intfloat/multilingual-e5-large-instruct'
        print(f"\nLoading {MODEL}...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL)
        model     = AutoModel.from_pretrained(
            MODEL, torch_dtype=torch.float16
        ).to('cuda')
        model.eval()
        EMB_DIM = model.config.hidden_size
        print(f"✅ Loaded! Hidden dim={EMB_DIM}")

        def last_token_pool(
            last_hidden, attention_mask
        ):
            seq_len = attention_mask.sum(dim=1) - 1
            batch   = last_hidden.shape[0]
            return last_hidden[
                torch.arange(
                    batch, device=last_hidden.device
                ),
                seq_len
            ]

        def encode(inputs, max_len=512, bs=16):
            all_embs = []
            for i in range(0, len(inputs), bs):
                batch = inputs[i:i+bs]
                enc   = tokenizer(
                    batch,
                    max_length=max_len,
                    padding=True,
                    truncation=True,
                    return_tensors='pt'
                ).to('cuda')
                with torch.no_grad():
                    out  = model(**enc)
                    embs = last_token_pool(
                        out.last_hidden_state,
                        enc['attention_mask']
                    )
                    embs = F.normalize(
                        embs.float(), p=2, dim=1
                    )
                all_embs.append(embs.cpu().numpy())
                del enc, out, embs
                torch.cuda.empty_cache()
            return np.vstack(all_embs)

        n_chunks = (n_total+CHUNK_SIZE-1) // CHUNK_SIZE

        for cid in range(
            start_idx // CHUNK_SIZE, n_chunks
        ):
            cs  = cid * CHUNK_SIZE
            ce  = min(cs + CHUNK_SIZE, n_total)
            cpath = os.path.join(
                MAL_CKPT_DIR, f'chunk_{cid}.npz'
            )

            if os.path.exists(cpath):
                print(f"  ⏭️  Chunk {cid} skip")
                continue

            print(f"\n── Chunk {cid}/{n_chunks-1} "
                  f"({cs}-{ce}) ──")

            c_texts = texts[cs:ce]
            c_refs  = ref_texts[cs:ce]

            # Essay embeddings
            print("  Encoding essays...")
            essay_embs = encode(
                [ESSAY_INSTR + t[:500]
                 for t in c_texts],
                max_len=512, bs=16
            )

            # Reference embeddings
            print("  Encoding references...")
            ref_embs = encode(
                [REF_INSTR + r[:500]
                 for r in c_refs],
                max_len=512, bs=16
            )

            # ── KEY FEATURE: Essay-Ref Similarity ──────
            # For each essay, compute similarity with
            # its OWN reference answer
            essay_ref_sims = np.array([
                float(cosine_similarity(
                    essay_embs[i].reshape(1,-1),
                    ref_embs[i].reshape(1,-1)
                )[0][0])
                for i in range(len(c_texts))
            ])

            # Sentence coherence
            print("  Computing coherence...")
            from nltk.tokenize import sent_tokenize
            # For Malayalam, split by punctuation
            import re
            chunk_coh = []
            for text in tqdm(c_texts):
                sents = [
                    s.strip() for s in
                    re.split(r'[.?!।]', text)
                    if s.strip() and len(s.split())>2
                ][:15]
                if len(sents) < 2:
                    chunk_coh.append([0.0, 0.0, 0.0])
                    continue
                sent_embs = encode(
                    [SENT_INSTR + s for s in sents],
                    max_len=128, bs=32
                )
                sims = [
                    float(cosine_similarity(
                        sent_embs[i].reshape(1,-1),
                        sent_embs[i+1].reshape(1,-1)
                    )[0][0])
                    for i in range(len(sent_embs)-1)
                ]
                chunk_coh.append([
                    float(np.mean(sims)),
                    float(np.min(sims)),
                    float(np.std(sims))
                ])

            np.savez_compressed(
                cpath,
                essay_embs     = essay_embs,
                ref_embs       = ref_embs,
                essay_ref_sims = essay_ref_sims,
                coherence      = np.array(chunk_coh)
            )
            print(f"  💾 Chunk {cid} saved!")
            print(f"  GPU: "
                  f"{torch.cuda.mem_get_info()[0]/1024**3:.1f}GB")

        print("\n✅ All chunks done!")

    # Assemble
    print("\nAssembling...")
    n_chunks  = (n_total+CHUNK_SIZE-1) // CHUNK_SIZE
    all_done  = sorted([
        int(f.replace('chunk_','').replace('.npz',''))
        for f in os.listdir(MAL_CKPT_DIR)
        if f.startswith('chunk_') and f.endswith('.npz')
    ])
    missing = [
        i for i in range(n_chunks)
        if i not in all_done
    ]

    if missing:
        print(f"❌ Missing chunks: {missing}")
    else:
        all_essay = []
        all_coh   = []
        all_sims  = []

        for cid in sorted(all_done):
            data = np.load(os.path.join(
                MAL_CKPT_DIR, f'chunk_{cid}.npz'
            ))
            all_essay.append(data['essay_embs'])
            all_coh.append(data['coherence'])
            all_sims.append(data['essay_ref_sims'])

        all_essay = np.vstack(all_essay)
        all_coh   = np.vstack(all_coh)
        all_sims  = np.concatenate(all_sims)
        EMB_DIM   = all_essay.shape[1]

        coh_df = pd.DataFrame(
            all_coh,
            columns=['mean_coherence',
                     'min_coherence','std_coherence']
        )
        sim_df = pd.DataFrame(
            {'essay_ref_similarity': all_sims}
        )
        emb_df = pd.DataFrame(
            all_essay,
            columns=[f'emb_{i}' for i in range(EMB_DIM)]
        )
        out = pd.concat([coh_df, sim_df, emb_df], axis=1)
        out.to_csv(MAL_EMB_PATH, index=False)
        print(f"💾 Saved: {out.shape}")
        print("  essay_ref_similarity stats:")
        print(f"  mean={all_sims.mean():.3f}  "
              f"std={all_sims.std():.3f}  "
              f"min={all_sims.min():.3f}  "
              f"max={all_sims.max():.3f}")
        print("\n✅ Step 4 Complete!")
        print("⚠️  Click Save Version NOW!")

✅ Embeddings exist!
Shape: (3000, 1028)


In [4]:
# ============================================================
# STEP 5: Assemble Full Feature Matrix
# ============================================================

MAL_FULL_PATH = os.path.join(
    SAVE_DIR, 'mal_full_features.parquet'
)

if os.path.exists(MAL_FULL_PATH):
    try:
        train_mal = pd.read_parquet(MAL_FULL_PATH)
        print(f"✅ Loaded: {train_mal.shape}")
    except Exception:
        os.remove(MAL_FULL_PATH)
        print("Corrupted — rebuilding...")

if not os.path.exists(MAL_FULL_PATH):
    emb_df = pd.read_csv(MAL_EMB_PATH)
    EMB_DIM = len([
        c for c in emb_df.columns
        if c.startswith('emb_')
    ])

    # Fix dtypes
    float_cols = (
        ['mean_coherence','min_coherence',
         'std_coherence','essay_ref_similarity'] +
        [f'emb_{i}' for i in range(EMB_DIM)]
    )
    for col in float_cols:
        if col in emb_df.columns:
            emb_df[col] = pd.to_numeric(
                emb_df[col], errors='coerce'
            ).astype(np.float32).fillna(0.0)

    META_COLS = [
        'essay_id','prompt_id','essay',
        'essay_score','ref_answer','prompt','max_marks'
    ]
    SF_COLS = [
        'word_count','sent_count','para_count',
        'avg_sent_len','sent_len_std','avg_word_len',
        'type_token_ratio','malayalam_ratio',
        'punct_diversity','char_count'
    ]
    EMB_COLS = float_cols  # all computed columns

    sf_df     = pd.read_csv(
        os.path.join(SAVE_DIR,
                     'mal_shallow_features.csv')
    )

    train_mal = pd.concat([
        df[META_COLS].reset_index(drop=True),
        sf_df[SF_COLS].reset_index(drop=True),
        emb_df[EMB_COLS].reset_index(drop=True),
    ], axis=1)

    train_mal.to_parquet(MAL_FULL_PATH, index=False)
    print(f"✅ Saved: {train_mal.shape}")
    print("⚠️  Click Save Version NOW!")

✅ Loaded: (3000, 1045)


In [5]:
# ============================================================
# STEP 6: XGBoost 10-Fold CV — Malayalam AES
# Strategy: group by prompt_id for fair evaluation
# Key feature: essay_ref_similarity
# ============================================================

MAL_RESULTS_PATH = os.path.join(
    SAVE_DIR, 'results_malayalam.csv'
)

if os.path.exists(MAL_RESULTS_PATH):
    print("✅ Results exist!")
    results_df = pd.read_csv(MAL_RESULTS_PATH)
    print(results_df.describe().round(3))

else:
    print("Loading feature matrix...")
    train_mal = pd.read_parquet(MAL_FULL_PATH)
    print(f"✅ Loaded: {train_mal.shape}")

    SF_COLS = [
        'word_count','sent_count','para_count',
        'avg_sent_len','sent_len_std','avg_word_len',
        'type_token_ratio','malayalam_ratio',
        'punct_diversity','char_count'
    ]
    EMB_DIM = len([
        c for c in train_mal.columns
        if c.startswith('emb_')
    ])
    EMB_COLS = (
        ['mean_coherence','min_coherence',
         'std_coherence',
         'essay_ref_similarity'] +  # ← KEY FEATURE
        [f'emb_{i}' for i in range(EMB_DIM)]
    )
    ALL_FEAT = SF_COLS + EMB_COLS

    X = np.nan_to_num(
        train_mal[ALL_FEAT].values.astype(np.float32),
        nan=0.0, posinf=0.0, neginf=0.0
    )
    y = train_mal['essay_score'].values.astype(
        np.float32
    )
    prompt_ids = train_mal['prompt_id'].values

    print(f"\nFeature matrix: {X.shape}")
    print(f"Score range: {y.min():.0f}-{y.max():.0f}")

    def qwk(y_true, y_pred):
        return cohen_kappa_score(
            y_true, y_pred, weights='quadratic'
        )
    def exact_agreement(y_true, y_pred):
        return np.mean(np.array(y_true)==np.array(y_pred))
    def adjacent_agreement(y_true, y_pred):
        return np.mean(
            np.abs(np.array(y_true)-np.array(y_pred))<=1
        )
    def spearman_r(y_true, y_pred):
        r, _ = spearmanr(y_true, y_pred)
        return r
    def clip_round(y_pred, y_min, y_max):
        return np.clip(
            np.round(y_pred).astype(int), y_min, y_max
        )

    # Score range is 0-10 for all prompts
    Y_MIN, Y_MAX   = 0, 10
    SCORE_RANGE    = Y_MAX - Y_MIN

    # XGBoost params
    PARAMS = dict(
        n_estimators=800,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.7,
        min_child_weight=3,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        device='cuda',
        verbosity=0
    )

    print("\n" + "="*65)
    print("  10-FOLD CV — Malayalam AES")
    print("  Model: multilingual-e5-large-instruct")
    print("  Key feature: essay_ref_similarity")
    print("="*65)

    kf = KFold(
        n_splits=10, shuffle=True, random_state=42
    )

    fold_qwk, fold_ea, fold_aa, fold_sp = [], [], [], []

    for fold_num, (tr_idx, te_idx) in enumerate(
        kf.split(X), 1
    ):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        y_tr_norm = y_tr / SCORE_RANGE

        m = XGBRegressor(**PARAMS)
        m.fit(X_tr, y_tr_norm)

        p      = m.predict(X_te)
        y_pred = clip_round(
            p * SCORE_RANGE, Y_MIN, Y_MAX
        )
        y_true = y_te.astype(int)

        fold_qwk.append(qwk(y_true, y_pred))
        fold_ea.append(exact_agreement(y_true, y_pred))
        fold_aa.append(adjacent_agreement(y_true, y_pred))
        fold_sp.append(spearman_r(y_true, y_pred))

        print(f"  Fold {fold_num:2d} → "
              f"QWK={fold_qwk[-1]:.3f}  "
              f"EA={fold_ea[-1]:.3f}  "
              f"AA={fold_aa[-1]:.3f}  "
              f"Sp={fold_sp[-1]:.3f}")

    mean_qwk = np.mean(fold_qwk)
    mean_ea  = np.mean(fold_ea)
    mean_aa  = np.mean(fold_aa)
    mean_sp  = np.mean(fold_sp)

    print(f"\n{'='*65}")
    print(f"  MEAN → QWK={mean_qwk:.3f}  "
          f"EA={mean_ea:.3f}  "
          f"AA={mean_aa:.3f}  "
          f"Spearman={mean_sp:.3f}")

    results_df = pd.DataFrame([{
        'n_essays'     : len(train_mal),
        'n_prompts'    : train_mal['prompt_id'].nunique(),
        'n_features'   : X.shape[1],
        'mean_qwk'     : round(mean_qwk, 3),
        'mean_ea'      : round(mean_ea,  3),
        'mean_aa'      : round(mean_aa,  3),
        'mean_spearman': round(mean_sp,  3),
        'model'        : 'multilingual-e5-large-instruct',
    }])

    results_df.to_csv(MAL_RESULTS_PATH, index=False)

    # Feature importance
    m_full = XGBRegressor(**PARAMS)
    m_full.fit(X, y / SCORE_RANGE)
    importance = m_full.feature_importances_
    feat_imp   = pd.DataFrame({
        'feature'   : ALL_FEAT,
        'importance': importance
    }).sort_values('importance', ascending=False)

    print(f"\n  Top 15 Features:")
    print(feat_imp.head(15).to_string(index=False))

    print(f"\n{'='*65}")
    print(f"  MALAYALAM AES RESULTS")
    print(f"  Essays: {len(train_mal)}")
    print(f"  Prompts: {train_mal['prompt_id'].nunique()}")
    print(f"  QWK:  {mean_qwk:.3f}")
    print(f"  EA:   {mean_ea:.3f}")
    print(f"  AA:   {mean_aa:.3f}")
    print(f"  Spe:  {mean_sp:.3f}")
    print(f"\n  essay_ref_similarity importance rank: "
          f"{list(feat_imp['feature']).index('essay_ref_similarity')+1}")

    print("\n✅ Malayalam AES Complete!")
    print("⚠️  Click Save Version NOW!")

✅ Results exist!
       n_essays  n_prompts  n_features  mean_qwk  mean_ea  mean_aa  \
count       1.0        1.0         1.0     1.000    1.000     1.00   
mean     3000.0      150.0      1038.0     0.932    0.574     0.89   
std         NaN        NaN         NaN       NaN      NaN      NaN   
min      3000.0      150.0      1038.0     0.932    0.574     0.89   
25%      3000.0      150.0      1038.0     0.932    0.574     0.89   
50%      3000.0      150.0      1038.0     0.932    0.574     0.89   
75%      3000.0      150.0      1038.0     0.932    0.574     0.89   
max      3000.0      150.0      1038.0     0.932    0.574     0.89   

       mean_spearman  
count          1.000  
mean           0.938  
std              NaN  
min            0.938  
25%            0.938  
50%            0.938  
75%            0.938  
max            0.938  


In [6]:
# ============================================================
# MALAYALAM AES — WORD COUNT FIX
# Replace raw word_count with ratio to reference length
# Also add ref_word_count as explicit feature
#
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score
from xgboost import XGBRegressor
from scipy.stats import spearmanr
import re, os, warnings
warnings.filterwarnings('ignore')

SAVE_DIR = '/kaggle/working/'

# Load data
df    = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/mal_dataset.csv'
)
q_df  = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/questions_150.csv'
)
df    = df.merge(
    q_df[['prompt_id','prompt','max_marks',
          'ref_answer']],
    on='prompt_id', how='left'
)

# ── IMPROVED SHALLOW FEATURES ─────────────────────────────
def get_mal_shallow_v2(essay_text, ref_text=None):
    """
    V2: Adds reference-normalized features.
    Key fix: word count ratio instead of raw count.
    """
    essay_text = str(essay_text)
    ref_text   = str(ref_text) if ref_text else ''

    words   = essay_text.split()
    n_words = max(len(words), 1)
    sents   = [
        s.strip() for s in
        re.split(r'[.?!।]', essay_text)
        if s.strip()
    ]
    n_sents = max(len(sents), 1)
    paras   = [
        p.strip() for p in essay_text.split('\n')
        if p.strip()
    ]
    mal_chars = sum(
        1 for c in essay_text
        if '\u0D00' <= c <= '\u0D7F'
    )
    total_alpha = sum(
        1 for c in essay_text if c.isalnum()
    )
    ttr = len(set(words)) / n_words
    sent_lens = [len(s.split()) for s in sents]

    # Reference-based features
    ref_words   = ref_text.split()
    n_ref_words = max(len(ref_words), 1)
    ref_sents   = [
        s.strip() for s in
        re.split(r'[.?!।]', ref_text)
        if s.strip()
    ]
    n_ref_sents = max(len(ref_sents), 1)

    # ── KEY FIX: Normalize by reference length ────────
    word_ratio = n_words / n_ref_words   # 0-1+
    sent_ratio = n_sents / n_ref_sents   # 0-1+

    return {
        # Raw counts (still useful)
        'word_count'         : n_words,
        'sent_count'         : n_sents,
        'para_count'         : max(len(paras), 1),
        'char_count'         : len(essay_text),

        # Reference-normalized (KEY FIX)
        'word_ratio'         : round(word_ratio, 3),
        'sent_ratio'         : round(sent_ratio, 3),
        'ref_word_count'     : n_ref_words,

        # Style features
        'avg_sent_len'       : n_words/n_sents,
        'sent_len_std'       : np.std(sent_lens)
                               if len(sent_lens)>1 else 0,
        'avg_word_len'       : np.mean([len(w)
                               for w in words])
                               if words else 0,
        'type_token_ratio'   : ttr,
        'malayalam_ratio'    : mal_chars/max(total_alpha,1),
        'punct_diversity'    : len(set(
            c for c in essay_text
            if not c.isalnum() and not c.isspace()
        )),
    }

# Compute improved features
print("Computing improved shallow features...")
from tqdm import tqdm

sf_v2 = [
    get_mal_shallow_v2(
        row['essay'], row['ref_answer_y']
    )
    for _, row in tqdm(df.iterrows(), total=len(df))
]
sf_v2_df = pd.DataFrame(sf_v2)
sf_v2_df.to_csv(
    os.path.join(SAVE_DIR, 'mal_shallow_v2.csv'),
    index=False
)
print(f"✅ V2 features: {sf_v2_df.shape}")

SF_COLS_V2 = [
    'word_count','sent_count','para_count','char_count',
    'word_ratio','sent_ratio','ref_word_count',
    'avg_sent_len','sent_len_std','avg_word_len',
    'type_token_ratio','malayalam_ratio','punct_diversity'
]

# ── Rebuild feature matrix with V2 shallow ─────────────
train_mal = pd.read_parquet(
    os.path.join(SAVE_DIR, 'mal_full_features.parquet')
)
EMB_DIM  = len([
    c for c in train_mal.columns
    if c.startswith('emb_')
])
EMB_COLS = (
    ['mean_coherence','min_coherence',
     'std_coherence','essay_ref_similarity'] +
    [f'emb_{i}' for i in range(EMB_DIM)]
)

# Replace shallow features with V2
ALL_FEAT_V2 = SF_COLS_V2 + EMB_COLS

train_v2 = pd.concat([
    df[['essay_id','prompt_id','essay',
        'essay_score','ref_answer_y']].reset_index(
        drop=True
    ),
    sf_v2_df[SF_COLS_V2].reset_index(drop=True),
    train_mal[EMB_COLS].reset_index(drop=True),
], axis=1)

train_v2.to_parquet(
    os.path.join(SAVE_DIR, 'mal_full_v2.parquet'),
    index=False
)
print(f"✅ V2 feature matrix: {train_v2.shape}")

# ── Run 10-fold CV with V2 features ───────────────────
X_v2 = np.nan_to_num(
    train_v2[ALL_FEAT_V2].values.astype(np.float32),
    nan=0.0, posinf=0.0, neginf=0.0
)
y    = train_v2['essay_score'].values.astype(np.float32)
Y_MIN, Y_MAX = 0, 10

PARAMS = dict(
    n_estimators=800, learning_rate=0.03,
    max_depth=6, subsample=0.8,
    colsample_bytree=0.7, min_child_weight=3,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1,
    tree_method='hist', device='cuda', verbosity=0
)

kf = KFold(n_splits=10, shuffle=True, random_state=42)
fold_qwk, fold_ea, fold_aa, fold_sp = [], [], [], []

print("\n" + "="*60)
print("  10-FOLD CV — V2 (Word Ratio Normalized)")
print("="*60)

for fold_num, (tr_idx, te_idx) in enumerate(
    kf.split(X_v2), 1
):
    X_tr, X_te = X_v2[tr_idx], X_v2[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    m = XGBRegressor(**PARAMS)
    m.fit(X_tr, y_tr/(Y_MAX-Y_MIN))
    p     = m.predict(X_te)
    y_pred = np.clip(
        np.round(p*(Y_MAX-Y_MIN)).astype(int),
        Y_MIN, Y_MAX
    )
    y_true = y_te.astype(int)

    q  = cohen_kappa_score(
        y_true, y_pred, weights='quadratic'
    )
    ea = np.mean(y_true==y_pred)
    aa = np.mean(np.abs(y_true-y_pred)<=1)
    r, _ = spearmanr(y_true, y_pred)

    fold_qwk.append(q)
    fold_ea.append(ea)
    fold_aa.append(aa)
    fold_sp.append(r)

    print(f"  Fold {fold_num:2d} → "
          f"QWK={q:.3f}  EA={ea:.3f}  "
          f"AA={aa:.3f}  Sp={r:.3f}")

mean_qwk = np.mean(fold_qwk)
mean_ea  = np.mean(fold_ea)
mean_aa  = np.mean(fold_aa)
mean_sp  = np.mean(fold_sp)

print(f"\n  {'='*52}")
print(f"  RESULTS COMPARISON:")
print(f"  V1 (raw word_count): 0.932 QWK")
print(f"  V2 (word_ratio):     {mean_qwk:.3f} QWK")
print(f"  EA: {mean_ea:.3f}  AA: {mean_aa:.3f}  "
      f"Spe: {mean_sp:.3f}")

# Feature importance
m_full = XGBRegressor(**PARAMS)
m_full.fit(X_v2, y/(Y_MAX-Y_MIN))
importance = m_full.feature_importances_
feat_imp   = pd.DataFrame({
    'feature'   : ALL_FEAT_V2,
    'importance': importance
}).sort_values('importance', ascending=False)

print(f"\n  Top 10 Features (V2):")
for _, row in feat_imp.head(10).iterrows():
    print(f"    {row['feature']:<25} "
          f"{row['importance']:.4f}")

print(f"\n  word_ratio rank: "
      f"{list(feat_imp['feature']).index('word_ratio')+1}"
      f" / {len(ALL_FEAT_V2)}")
print(f"  word_count rank: "
      f"{list(feat_imp['feature']).index('word_count')+1}"
      f" / {len(ALL_FEAT_V2)}")
print(f"\n  essay_ref_similarity importance rank: "
          f"{list(feat_imp['feature']).index('essay_ref_similarity')+1}")

# Train final model on all data
model_xgb_v2 = XGBRegressor(**PARAMS)
model_xgb_v2.fit(X_v2, y/(Y_MAX-Y_MIN))
print("\n✅ V2 model trained!")

Computing improved shallow features...


100%|██████████| 3000/3000 [00:01<00:00, 2412.35it/s]


✅ V2 features: (3000, 13)
✅ V2 feature matrix: (3000, 1046)

  10-FOLD CV — V2 (Word Ratio Normalized)
  Fold  1 → QWK=0.935  EA=0.607  AA=0.917  Sp=0.931
  Fold  2 → QWK=0.940  EA=0.567  AA=0.900  Sp=0.945
  Fold  3 → QWK=0.947  EA=0.543  AA=0.913  Sp=0.951
  Fold  4 → QWK=0.943  EA=0.610  AA=0.893  Sp=0.949
  Fold  5 → QWK=0.951  EA=0.593  AA=0.927  Sp=0.958
  Fold  6 → QWK=0.938  EA=0.587  AA=0.907  Sp=0.942
  Fold  7 → QWK=0.958  EA=0.603  AA=0.947  Sp=0.958
  Fold  8 → QWK=0.947  EA=0.620  AA=0.907  Sp=0.953
  Fold  9 → QWK=0.935  EA=0.593  AA=0.937  Sp=0.928
  Fold 10 → QWK=0.933  EA=0.583  AA=0.920  Sp=0.939

  RESULTS COMPARISON:
  V1 (raw word_count): 0.932 QWK
  V2 (word_ratio):     0.943 QWK
  EA: 0.591  AA: 0.917  Spe: 0.945

  Top 10 Features (V2):
    sent_count                0.1405
    word_ratio                0.0558
    word_count                0.0175
    char_count                0.0103
    sent_ratio                0.0089
    std_coherence             0.0078
    em

In [14]:
# ============================================================
# MALAYALAM AEG
# Custom prompt + ref answer + student answer → score
# ============================================================

import os, re, warnings
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from xgboost import XGBRegressor
import pandas as pd
warnings.filterwarnings('ignore')

SAVE_DIR = '/kaggle/working/'

# ── Load embedding model FRESH ─────────────────────────────
MODEL_NAME = 'intfloat/multilingual-e5-large-instruct'
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
emb_model = AutoModel.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16
).to('cuda')
emb_model.eval()
print("✅ Model loaded!")

# ── Load dataset and train model ───────────────────────────
df    = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/mal_dataset.csv'
)
q_df  = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/questions_150.csv'
)
df    = df.merge(
    q_df[['prompt_id','prompt','max_marks']],
    on='prompt_id', how='left'
)
train_mal = pd.read_parquet(
    os.path.join(SAVE_DIR,'mal_full_v2.parquet')
)

SF_COLS_V2 = [
    'word_count','sent_count','para_count','char_count',
    'word_ratio','sent_ratio','ref_word_count',
    'avg_sent_len','sent_len_std','avg_word_len',
    'type_token_ratio','malayalam_ratio','punct_diversity'
]
EMB_DIM  = len([c for c in train_mal.columns
                if c.startswith('emb_')])
EMB_COLS = (
    ['mean_coherence','min_coherence',
     'std_coherence','essay_ref_similarity'] +
    [f'emb_{i}' for i in range(EMB_DIM)]
)
ALL_FEAT = SF_COLS_V2 + EMB_COLS

X = np.nan_to_num(
    train_mal[ALL_FEAT].values.astype(np.float32),
    nan=0.0, posinf=0.0, neginf=0.0
)
y = train_mal['essay_score'].values.astype(np.float32)
Y_MIN, Y_MAX = 0, 10

PARAMS = dict(
    n_estimators=800, learning_rate=0.03,
    max_depth=6, subsample=0.8,
    colsample_bytree=0.7, min_child_weight=3,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1,
    tree_method='hist', device='cuda', verbosity=0
)
model_xgb_v2 = XGBRegressor(**PARAMS)
model_xgb_v2.fit(X, y/(Y_MAX-Y_MIN))
print("✅ XGBoost trained on full data!")

# ── Instructions ───────────────────────────────────────────
ESSAY_INSTR = (
    "Instruct: Represent this Malayalam student "
    "answer for evaluating content accuracy and "
    "completeness against a reference answer\nQuery: "
)
REF_INSTR = (
    "Instruct: Represent this Malayalam reference "
    "answer as the gold standard for scoring\nQuery: "
)
SENT_INSTR = (
    "Instruct: Represent this Malayalam sentence "
    "for measuring coherence\nQuery: "
)

# ── Helper functions ───────────────────────────────────────
def last_token_pool(last_hidden, attention_mask):
    seq_len = attention_mask.sum(dim=1) - 1
    batch   = last_hidden.shape[0]
    return last_hidden[
        torch.arange(batch, device=last_hidden.device),
        seq_len
    ]

def get_embedding(text, instruction):
    enc = tokenizer(
        [instruction + str(text)[:500]],
        max_length=512, padding=True,
        truncation=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        out = emb_model(**enc)
        emb = last_token_pool(
            out.last_hidden_state,
            enc['attention_mask']
        )
        emb = F.normalize(emb.float(), p=2, dim=1)
    return emb.cpu().numpy()[0]

def get_mal_shallow_v2(essay_text, ref_text):
    essay_text = str(essay_text)
    ref_text   = str(ref_text)
    words      = essay_text.split()
    n_words    = max(len(words), 1)
    sents      = [
        s.strip() for s in
        re.split(r'[.?!\u0964]', essay_text)
        if s.strip()
    ]
    n_sents    = max(len(sents), 1)
    paras      = [
        p.strip() for p in essay_text.split('\n')
        if p.strip()
    ]
    mal_chars  = sum(
        1 for c in essay_text
        if '\u0D00' <= c <= '\u0D7F'
    )
    total_alph = sum(1 for c in essay_text if c.isalnum())
    ttr        = len(set(words)) / n_words
    sent_lens  = [len(s.split()) for s in sents]
    ref_words  = ref_text.split()
    n_ref_w    = max(len(ref_words), 1)
    ref_sents  = [
        s.strip() for s in
        re.split(r'[.?!\u0964]', ref_text)
        if s.strip()
    ]
    n_ref_s    = max(len(ref_sents), 1)
    return {
        'word_count'      : n_words,
        'sent_count'      : n_sents,
        'para_count'      : max(len(paras), 1),
        'char_count'      : len(essay_text),
        'word_ratio'      : round(n_words/n_ref_w, 3),
        'sent_ratio'      : round(n_sents/n_ref_s, 3),
        'ref_word_count'  : n_ref_w,
        'avg_sent_len'    : n_words/n_sents,
        'sent_len_std'    : np.std(sent_lens)
                            if len(sent_lens)>1 else 0,
        'avg_word_len'    : np.mean([len(w)
                            for w in words])
                            if words else 0,
        'type_token_ratio': ttr,
        'malayalam_ratio' : mal_chars/max(total_alph,1),
        'punct_diversity' : len(set(
            c for c in essay_text
            if not c.isalnum() and not c.isspace()
        )),
    }

def get_coherence(text):
    sents = [
        s.strip() for s in
        re.split(r'[.?!\u0964]', str(text))
        if s.strip() and len(s.split()) > 2
    ][:15]
    if len(sents) < 2:
        return 0.0, 0.0, 0.0
    embs = np.array([
        get_embedding(s, SENT_INSTR)
        for s in sents
    ])
    sims = [
        float(cosine_similarity(
            embs[i].reshape(1,-1),
            embs[i+1].reshape(1,-1)
        )[0][0])
        for i in range(len(embs)-1)
    ]
    return (float(np.mean(sims)),
            float(np.min(sims)),
            float(np.std(sims)))


# ── MAIN PREDICTION FUNCTION ───────────────────────────────
def predict_v2_custom(
    prompt_text,
    ref_answer,
    student_answer,
    max_marks=10,
    actual_score=None
):
    """
    Predict score for a custom Malayalam prompt.
    Prints full prompt, ref answer, student answer,
    computed features, and predicted score.
    """
    print("\n" + "="*65)
    print("  MALAYALAM ASAG — V2 CUSTOM PREDICTION")
    print("="*65)

    # Print full texts for report
    print(f"\n{'─'*65}")
    print("  QUESTION / PROMPT:")
    print(f"{'─'*65}")
    print(f"  {prompt_text.strip()}")

    print(f"\n{'─'*65}")
    print("  REFERENCE ANSWER (Model Answer):")
    print(f"{'─'*65}")
    # Print reference in wrapped lines
    words_ref = ref_answer.strip().split()
    line = "  "
    for w in words_ref:
        if len(line) + len(w) > 70:
            print(line)
            line = "  " + w + " "
        else:
            line += w + " "
    if line.strip():
        print(line)

    print(f"\n{'─'*65}")
    print("  STUDENT ANSWER:")
    print(f"{'─'*65}")
    words_stu = student_answer.strip().split()
    line = "  "
    for w in words_stu:
        if len(line) + len(w) > 70:
            print(line)
            line = "  " + w + " "
        else:
            line += w + " "
    if line.strip():
        print(line)

    print(f"\n{'─'*65}")
    print("  COMPUTING FEATURES...")
    print(f"{'─'*65}")

    # Features
    sf = get_mal_shallow_v2(student_answer, ref_answer)
    print(f"  word_count     : {sf['word_count']}")
    print(f"  sent_count     : {sf['sent_count']}")
    print(f"  ref_word_count : {sf['ref_word_count']}")
    print(f"  word_ratio (V2): {sf['word_ratio']:.3f} "
          f"({sf['word_count']}/{sf['ref_word_count']})")
    print(f"  type_token_ratio: {sf['type_token_ratio']:.3f}")

    essay_emb = get_embedding(student_answer, ESSAY_INSTR)
    ref_emb   = get_embedding(ref_answer,     REF_INSTR)
    er_sim    = float(cosine_similarity(
        essay_emb.reshape(1,-1),
        ref_emb.reshape(1,-1)
    )[0][0])
    print(f"  essay_ref_sim  : {er_sim:.4f}")

    coh_m, coh_min, coh_std = get_coherence(student_answer)
    print(f"  coherence(mean): {coh_m:.4f}")
    print(f"  coherence(std) : {coh_std:.4f}")

    # Feature vector
    sf_vec  = [sf[c] for c in SF_COLS_V2]
    emb_vec = (
        [coh_m, coh_min, coh_std, er_sim] +
        essay_emb.tolist()
    )
    feat    = np.array(
        sf_vec + emb_vec, dtype=np.float32
    ).reshape(1,-1)
    feat    = np.nan_to_num(
        feat, nan=0.0, posinf=0.0, neginf=0.0
    )

    p_norm  = model_xgb_v2.predict(feat)[0]
    p_raw   = p_norm * (Y_MAX - Y_MIN)
    p_score = int(np.clip(
        np.round(p_raw), Y_MIN, Y_MAX
    ))

    print(f"\n{'═'*65}")
    print(f"  PREDICTED SCORE : {p_score} / {max_marks}")
    if actual_score is not None:
        diff   = p_score - actual_score
        status = "✅ Correct" if abs(diff)<=1 else "⚠️ Off"
        print(f"  ACTUAL SCORE    : {actual_score} / {max_marks}")
        print(f"  DIFFERENCE      : {diff:.1f}  {status}")
    print(f"{'═'*65}\n")
    return p_score


# ════════════════════════════════════════════════════════════
# CUSTOM TESTS — Add your own prompts here
# ════════════════════════════════════════════════════════════

# ── TEST 1 ───────────────────────────────────────────────
predict_v2_custom(
    prompt_text = "ക്വിറ്റ് ഇന്ത്യ സമരത്തെക്കുറിച്ച് വിശദീകരിക്കുക.",

    ref_answer  = """സ്വാതന്ത്ര്യത്തിനായുള്ള ഭാരതീയരുടെ പോരാട്ടത്തിന്റെ ഗതി
മാറ്റിയ ദിനമാണ് ക്വിറ്റ് ഇന്ത്യാ ദിനം. ബ്രിട്ടീഷുകാർ ഇന്ത്യ
വിടണം എന്ന മുദ്രാവാക്യം മുഴക്കി ഗാന്ധിജിയുടെ നേതൃത്വത്തിൽ
1942 ആഗസ്റ്റ് 9-ന് ആരംഭിച്ച ബഹുജന മുന്നേറ്റമാണ് ഈ സമരം.
ഓഗസ്റ്റ് 8-ന് ബോംബെ സമ്മേളനത്തിൽ ഗാന്ധിജി
"ചെയ്യുക അല്ലെങ്കിൽ മരിക്കുക" എന്ന് ആഹ്വാനം ചെയ്തു.
ക്രിപ്സ് മിഷന്റെ പരാജയം ഈ സമരത്തിന്റെ
ഒരു പ്രധാന കാരണമായിരുന്നു.""",

    student_answer = """ക്വിറ്റ് ഇന്ത്യ സമരം 1942-ൽ ഗാന്ധിജിയുടെ നേതൃത്വത്തിൽ
ആരംഭിച്ചു. ബ്രിട്ടീഷുകാർ ഇന്ത്യ വിടണം എന്ന ആവശ്യവുമായി
നടന്ന ഈ സമരം ഇന്ത്യൻ സ്വാതന്ത്ര്യ സമരത്തിലെ
ഒരു നിർണ്ണായക ഘട്ടമായിരുന്നു. ഓഗസ്റ്റ് 8-ന് ബോംബെ
സമ്മേളനത്തിൽ ക്വിറ്റ് ഇന്ത്യ പ്രമേയം പാസാക്കി.
ഗാന്ധിജി "ചെയ്യുക അല്ലെങ്കിൽ മരിക്കുക" എന്ന്
ആഹ്വാനം ചെയ്തു. ക്രിപ്സ് മിഷൻ പരാജയപ്പെട്ടതാണ്
ഈ സമരത്തിന് കാരണമായത്. ഒരു ലക്ഷത്തോളം
ആളുകൾ അറസ്റ്റ് ചെയ്യപ്പെട്ടു.""",

    max_marks    = 10,
    actual_score = 8
)

# ── TEST 2 ───────────────────────────────────────────────
predict_v2_custom(
    prompt_text = "ഇന്ത്യൻ ഭരണഘടനയുടെ പ്രധാന സവിശേഷതകളെയും അത് രൂപപ്പെടുത്തിയ പശ്ചാത്തലത്തെയും കുറിച്ച് ഒരു ലഘു കുറിപ്പ് തയ്യാറാക്കുക.",

    ref_answer  = """ലോകത്തിലെ ഏറ്റവും വലിയ ലിഖിത ഭരണഘടനയാണ് ഇന്ത്യയുടേത്. 1949 നവംബർ 26-ന് ഭരണഘടനാ നിർമ്മാണ സഭ ഇത് അംഗീകരിക്കുകയും 1950 ജനുവരി 26-ന് നിലവിൽ വരികയും ചെയ്തു. ഡോ. ബി.ആർ. അംബേദ്കർ ആണ് ഭരണഘടനയുടെ മുഖ്യ ശില്പിയായി അറിയപ്പെടുന്നത്. ഒരു പരമാധികാര, സോഷ്യലിസ്റ്റ്, മതേതര, ജനാധിപത്യ, റിപ്പബ്ലിക് രാജ്യമായാണ് ഭരണഘടന ഇന്ത്യയെ വിഭാവനം ചെയ്യുന്നത്.

മൗലികാവകാശങ്ങൾ, മൗലിക കർത്തവ്യങ്ങൾ, നിർദ്ദേശക തത്വങ്ങൾ എന്നിവ ഭരണഘടനയുടെ സുപ്രധാന ഭാഗങ്ങളാണ്. പൗരന്മാരുടെ അവകാശങ്ങൾ സംരക്ഷിക്കുന്നതിനായി സ്വതന്ത്രമായ ഒരു നീതിന്യായ വ്യവസ്ഥയും ഇവിടെ നിലനിൽക്കുന്നു. പാർലമെന്ററി ഭരണരീതിയാണ് ഇന്ത്യ പിന്തുടരുന്നത്. ഭരണഘടന അനുശാസിക്കുന്ന പ്രകാരം പ്രായപൂർത്തി വോട്ടവകാശത്തിലൂടെയാണ് ജനപ്രതിനിധികളെ തിരഞ്ഞെടുക്കുന്നത്. ഭരണഘടനയിലെ ഭേദഗതികൾ വഴി കാലാനുസൃതമായ മാറ്റങ്ങൾ വരുത്താനും സാധിക്കും. ഫെഡറൽ സംവിധാനവും ഏകീകൃത സ്വഭാവവും ഒത്തുചേരുന്ന സവിശേഷമായ ഒരു ശൈലിയാണ് ഇതിനുള്ളത്. ഇന്ത്യയുടെ അഖണ്ഡതയും ഐക്യവും കാത്തുസൂക്ഷിക്കാൻ ഭരണഘടന ഓരോ പൗരനെയും പ്രാപ്തനാക്കുന്നു.
  """,

    student_answer = """  ഭാരതത്തിന്റെ പരമാധികാരവും നിയമവ്യവസ്ഥയും ഉറപ്പുവരുത്തുന്ന ഉന്നതമായ രേഖയാണ് ഇന്ത്യൻ ഭരണഘടന. ലോകത്തെ ഏറ്റവും ദീർഘമായ ലിഖിത ഭരണഘടനയാണിത്. 1950 ജനുവരി 26 മുതലാണ് ഇത് ഇന്ത്യയിൽ നടപ്പിലായത്. ഭരണഘടനാ ശില്പിയായ ഡോ. ബി.ആർ. അംബേദ്കറുടെ നേതൃത്വത്തിലാണ് ഇത് തയ്യാറാക്കിയത്. ഇന്ത്യയെ ഒരു മതേതര ജനാധിപത്യ രാജ്യമായി നിലനിർത്താൻ ഇത് സഹായിക്കുന്നു.

പൗരന്മാർക്ക് അന്തസ്സോടെ ജീവിക്കാൻ ആവശ്യമായ മൗലികാവകാശങ്ങൾ ഭരണഘടന നൽകുന്നുണ്ട്. അതോടൊപ്പം പൗരന്മാരുടെ കടമകളെക്കുറിച്ച് മൗലിക കർത്തവ്യങ്ങളിൽ പ്രതിപാദിക്കുന്നു. സർക്കാരുകൾ പാലിക്കേണ്ട നയങ്ങളെക്കുറിച്ച് നിർദ്ദേശക തത്വങ്ങൾ വ്യക്തമാക്കുന്നു. സ്വതന്ത്രമായ സുപ്രീം കോടതിയും ഹൈക്കോടതികളും പൗരന്മാരുടെ നീതി ഉറപ്പാക്കുന്നു. പ്രായപൂർത്തിയായ എല്ലാ പൗരന്മാർക്കും ജാതിമത ഭേദമന്യേ വോട്ട് ചെയ്യാനുള്ള അധികാരം ഇതിലൂടെ ലഭിക്കുന്നു. പാർലമെന്ററി ജനാധിപത്യം വഴി ജനങ്ങൾ നേരിട്ടാണ് ഭരണം തിരഞ്ഞെടുക്കുന്നത്. ആവശ്യമായ ഘട്ടങ്ങളിൽ മാറ്റങ്ങൾ വരുത്താൻ ഭരണഘടനാ ഭേദഗതികൾ സഹായിക്കുന്നു. ഏകീകൃതമായ പൗരത്വവും ശക്തമായ കേന്ദ്രസർക്കാരും ഇന്ത്യയുടെ സവിശേഷതയാണ്. നമ്മുടെ ജനാധിപത്യത്തിന്റെ നട്ടെല്ലാണ് ഈ ഭരണഘടന.
  """,

    max_marks    = 10,
    actual_score = 9.5
)


Loading intfloat/multilingual-e5-large-instruct...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ Model loaded!
✅ XGBoost trained on full data!

  MALAYALAM ASAG — V2 CUSTOM PREDICTION

─────────────────────────────────────────────────────────────────
  QUESTION / PROMPT:
─────────────────────────────────────────────────────────────────
  ക്വിറ്റ് ഇന്ത്യ സമരത്തെക്കുറിച്ച് വിശദീകരിക്കുക.

─────────────────────────────────────────────────────────────────
  REFERENCE ANSWER (Model Answer):
─────────────────────────────────────────────────────────────────
  സ്വാതന്ത്ര്യത്തിനായുള്ള ഭാരതീയരുടെ പോരാട്ടത്തിന്റെ ഗതി മാറ്റിയ 
  ദിനമാണ് ക്വിറ്റ് ഇന്ത്യാ ദിനം. ബ്രിട്ടീഷുകാർ ഇന്ത്യ വിടണം എന്ന 
  മുദ്രാവാക്യം മുഴക്കി ഗാന്ധിജിയുടെ നേതൃത്വത്തിൽ 1942 ആഗസ്റ്റ് 9-ന് 
  ആരംഭിച്ച ബഹുജന മുന്നേറ്റമാണ് ഈ സമരം. ഓഗസ്റ്റ് 8-ന് ബോംബെ 
  സമ്മേളനത്തിൽ ഗാന്ധിജി "ചെയ്യുക അല്ലെങ്കിൽ മരിക്കുക" എന്ന് ആഹ്വാനം 
  ചെയ്തു. ക്രിപ്സ് മിഷന്റെ പരാജയം ഈ സമരത്തിന്റെ ഒരു പ്രധാന 
  കാരണമായിരുന്നു. 

─────────────────────────────────────────────────────────────────
  STUDENT ANSWER:
────────────────────────────────────────────

9

In [15]:
import matplotlib.pyplot as plt
import numpy as np

# Define styling constants locally
GREY = "#F5F5F5"
TEAL = "#008080"
NAVY = "#000080"

# ════════════════════════════════════════════════════════════
# FIG 4 — Malayalam per-fold QWK (V2 - Word Ratio Normalized)
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(GREY)
ax.set_facecolor(GREY)

# Updated QWK values from V2 results
fold_qwk = [0.935, 0.940, 0.947, 0.943, 0.951, 
            0.938, 0.958, 0.947, 0.935, 0.933]
folds    = [f"Fold {i}" for i in range(1, 11)]
mean_q   = np.mean(fold_qwk)

bars = ax.bar(folds, fold_qwk, color=TEAL,
              edgecolor='white', linewidth=1, width=0.6)

# Horizontal mean line
ax.axhline(mean_q, color=NAVY, lw=2,
           linestyle='--',
           label=f"Mean QWK = {mean_q:.4f}")

# Axis limits and labels
ax.set_ylim(0.92, 0.97) 
ax.set_ylabel("QWK Score", fontsize=12, color=NAVY)

# Updated Title with aggregate metrics
ax.set_title("Malayalam AES V2 (Word Ratio Normalized) — QWK per Fold\n"
             "10-fold CV | Avg QWK: 0.943 | EA: 0.591 | AA: 0.916 | Sp: 0.943",
             fontsize=13, color=NAVY, fontweight='bold')

ax.legend(fontsize=11)

# Annotate bars
for bar, q in zip(bars, fold_qwk):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.0005,
            f"{q:.3f}", ha='center', va='bottom',
            fontsize=9, color=NAVY, fontweight='bold')

ax.grid(axis='y', alpha=0.3, color='gray')
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()

# SAVING TO WORKING DIRECTORY
# Removed {SAVE_DIR} to save in the current folder
plt.savefig('viz_malayalam_folds_v2.png',
            dpi=150, bbox_inches='tight', facecolor=GREY)
plt.close()

print(f"✅ Saved to working directory: viz_malayalam_folds_v2.png")

✅ Saved to working directory: viz_malayalam_folds_v2.png


In [16]:
# ============================================================
# FIX — Check column names and merge correctly
# ============================================================

import pandas as pd

df   = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/mal_dataset.csv'
)
q_df = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/questions_150.csv'
)

# ── Check actual column names ──────────────────────────────
print("mal_dataset.csv columns:")
print(df.columns.tolist())
print(f"\nShape: {df.shape}")
print(df.head(2).to_string())

print("\nquestions_150.csv columns:")
print(q_df.columns.tolist())
print(f"\nShape: {q_df.shape}")
print(q_df.head(2).to_string())

mal_dataset.csv columns:
['essay_id', 'prompt_id', 'essay', 'ref_answer', 'essay_score']

Shape: (3000, 5)
   essay_id  prompt_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [17]:
# ── Add this at the TOP of the cell ───────────────────────
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
print(f"✅ HF token loaded: {HF_TOKEN[:8]}...")

✅ HF token loaded: hf_RLmRm...


In [18]:
# ============================================================
# MALAYALAM MODEL COMPARISON STUDY
# Tests multiple embedding models on same dataset
# Gives paper a proper ablation/comparison table
# ============================================================

# Models to compare:
# 1. multilingual-e5-large-instruct (yours — baseline)
# 2. Qwen2.5-1.5B-Instruct (decoder LLM)
# 3. ai4bharat/indic-bert (encoder, Indian languages)
# 4. google/muril-base-cased (encoder, multilingual Indian)
# 5. sentence-transformers/LaBSE (encoder, 109 languages)

# This cell: compute embeddings for all models
# Then compare QWK in one table


import os, gc, re, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score
from xgboost import XGBRegressor
from scipy.stats import spearmanr
from tqdm import tqdm
warnings.filterwarnings('ignore')

SAVE_DIR = '/kaggle/working/'

# ── Load datasets ──────────────────────────────────────────
df_raw   = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/mal_dataset.csv'
)
q_df_raw = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/questions_150.csv'
)

# ref_answer is already in df_raw!
# Only take prompt text and max_marks from questions file
# Do NOT include ref_answer from q_df (avoids x/y conflict)
df = df_raw.merge(
    q_df_raw[['prompt_id', 'prompt', 'max_marks']],
    on='prompt_id',
    how='left'
)

print(f"✅ Dataset loaded: {df.shape}")
print(f"   Columns: {df.columns.tolist()}")
print(f"   ref_answer nulls: {df['ref_answer'].isna().sum()}")
print(f"   Essays: {len(df)}, Prompts: "
      f"{df['prompt_id'].nunique()}")

# Use ref_answer from mal_dataset.csv directly
texts     = df['essay'].astype(str).tolist()
ref_texts = df['ref_answer'].astype(str).tolist()
y_scores  = df['essay_score'].values.astype(np.float32)
n_total   = len(texts)
print(f"\n✅ Ready: {n_total} essays with reference answers")
print(f"   Sample ref[0][:80]: {ref_texts[0][:80]}...")

# ── V2 Shallow features ────────────────────────────────────
def get_sf_v2(essay, ref):
    essay = str(essay)
    ref   = str(ref)
    words = essay.split()
    n_w   = max(len(words), 1)
    sents = [s.strip() for s in
             re.split(r'[.?!\u0964]', essay)
             if s.strip()]
    n_s   = max(len(sents), 1)
    paras = [p.strip() for p in essay.split('\n')
             if p.strip()]
    ref_w = max(len(ref.split()), 1)
    ref_s = max(len([s for s in
              re.split(r'[.?!\u0964]', ref)
              if s.strip()]), 1)
    mal   = sum(1 for c in essay
                if '\u0D00' <= c <= '\u0D7F')
    alp   = max(sum(1 for c in essay if c.isalnum()), 1)
    sl    = [len(s.split()) for s in sents]
    return [
        n_w, n_s, max(len(paras), 1), len(essay),
        n_w / ref_w,         # word_ratio
        n_s / ref_s,         # sent_ratio
        ref_w,               # ref_word_count
        n_w / n_s,           # avg_sent_len
        float(np.std(sl)) if len(sl) > 1 else 0.0,
        float(np.mean([len(w) for w in words]))
             if words else 0.0,
        len(set(words)) / n_w,
        mal / alp,
        len(set(c for c in essay
                if not c.isalnum()
                and not c.isspace()))
    ]

print("\nComputing V2 shallow features...")
sf_all = np.array([
    get_sf_v2(e, r)
    for e, r in tqdm(zip(texts, ref_texts),
                     total=n_total)
], dtype=np.float32)
print(f"✅ Shallow features: {sf_all.shape}")


# ── Generic embedding function ─────────────────────────────
def embed_dataset(model_name, essay_instr,
                  ref_instr, sent_instr,
                  checkpoint_name,
                  use_cls=False,
                  batch_size=16,
                  max_len=512):
    CHUNK    = 500
    ckpt_dir = os.path.join(
        SAVE_DIR, f'mal_{checkpoint_name}_ckpts'
    )
    emb_path = os.path.join(
        SAVE_DIR, f'mal_emb_{checkpoint_name}.npz'
    )
    os.makedirs(ckpt_dir, exist_ok=True)

    if os.path.exists(emb_path):
        print(f"  ✅ {checkpoint_name} exists — loading...")
        d = np.load(emb_path)
        return (d['essay_embs'],
                d['er_sims'],
                d['coherence'])

    done = sorted([
        int(f.replace('chunk_','').replace('.npz',''))
        for f in os.listdir(ckpt_dir)
        if f.startswith('chunk_') and f.endswith('.npz')
    ])
    start = (max(done)+1)*CHUNK if done else 0

    print(f"\n  Loading {model_name}...")
    gc.collect()
    torch.cuda.empty_cache()

    tokenizer = AutoTokenizer.from_pretrained(
    model_name, trust_remote_code=True,
    token=HF_TOKEN          
    )
    model     = AutoModel.from_pretrained(
        model_name,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        token=HF_TOKEN         
    ).to('cuda')
    model.eval()
    print(f"  ✅ Loaded! "
          f"GPU: {torch.cuda.mem_get_info()[0]/1024**3:.1f}GB")

    def pool(out, mask):
        if use_cls:
            return out.last_hidden_state[:, 0, :]
        seq = mask.sum(dim=1) - 1
        b   = out.last_hidden_state.shape[0]
        return out.last_hidden_state[
            torch.arange(b, device=seq.device), seq
        ]

    def encode(texts_in, instr, bs=batch_size):
        inputs   = [instr + str(t)[:400]
                    for t in texts_in]
        all_embs = []
        for i in range(0, len(inputs), bs):
            batch = inputs[i:i+bs]
            enc   = tokenizer(
                batch, max_length=max_len,
                padding=True, truncation=True,
                return_tensors='pt'
            ).to('cuda')
            with torch.no_grad():
                out  = model(**enc)
                embs = pool(out, enc['attention_mask'])
                embs = F.normalize(
                    embs.float(), p=2, dim=1
                )
            all_embs.append(embs.cpu().numpy())
            del enc, out, embs
            torch.cuda.empty_cache()
        return np.vstack(all_embs)

    n_chunks = (n_total + CHUNK - 1) // CHUNK

    for cid in range(start // CHUNK, n_chunks):
        cs    = cid * CHUNK
        ce    = min(cs + CHUNK, n_total)
        cpath = os.path.join(
            ckpt_dir, f'chunk_{cid}.npz'
        )
        if os.path.exists(cpath):
            print(f"  ⏭  chunk {cid}")
            continue

        print(f"\n  ── chunk {cid}/{n_chunks-1} "
              f"({cs}–{ce}) ──")
        c_t = texts[cs:ce]
        c_r = ref_texts[cs:ce]

        # Essay embeddings
        print("  Encoding essays...")
        e_embs = encode(c_t, essay_instr)

        # Reference embeddings
        print("  Encoding references...")
        r_embs = encode(c_r, ref_instr)

        # Essay-Reference similarity
        er_sims = np.array([
            float(cosine_similarity(
                e_embs[i].reshape(1,-1),
                r_embs[i].reshape(1,-1)
            )[0][0])
            for i in range(len(c_t))
        ])
        print(f"  er_sim: mean={er_sims.mean():.3f}")

        # Sentence coherence
        coh = []
        for text in tqdm(c_t, desc="  coherence"):
            sents = [s.strip() for s in
                     re.split(r'[.?!\u0964]', text)
                     if s.strip()
                     and len(s.split()) > 2][:15]
            if len(sents) < 2:
                coh.append([0., 0., 0.])
                continue
            se   = encode(sents, sent_instr,
                          bs=min(32, len(sents)))
            sims = [float(cosine_similarity(
                se[i].reshape(1,-1),
                se[i+1].reshape(1,-1)
            )[0][0]) for i in range(len(se)-1)]
            coh.append([float(np.mean(sims)),
                        float(np.min(sims)),
                        float(np.std(sims))])

        np.savez_compressed(
            cpath,
            essay_embs = e_embs,
            er_sims    = er_sims,
            coherence  = np.array(coh)
        )
        print(f"  💾 chunk {cid} saved!")

    # ── Assemble all chunks ────────────────────────────────
    all_e, all_s, all_c = [], [], []
    for cid in range(n_chunks):
        d = np.load(os.path.join(
            ckpt_dir, f'chunk_{cid}.npz'
        ))
        all_e.append(d['essay_embs'])
        all_s.append(d['er_sims'])
        all_c.append(d['coherence'])

    essay_embs = np.vstack(all_e)
    er_sims    = np.concatenate(all_s)
    coherence  = np.vstack(all_c)

    np.savez_compressed(
        emb_path,
        essay_embs = essay_embs,
        er_sims    = er_sims,
        coherence  = coherence
    )
    print(f"\n  💾 {checkpoint_name} assembled: "
          f"{essay_embs.shape}")
    # Free GPU memory
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return essay_embs, er_sims, coherence


# ── Cross-validation function ──────────────────────────────
def run_cv(essay_embs, er_sims, coherence,
           sf, y, model_label):
    X = np.nan_to_num(
        np.hstack([
            sf,
            coherence,
            er_sims.reshape(-1, 1),
            essay_embs
        ]).astype(np.float32),
        nan=0.0, posinf=0.0, neginf=0.0
    )
    PARAMS = dict(
        n_estimators=800, learning_rate=0.03,
        max_depth=6, subsample=0.8,
        colsample_bytree=0.7, min_child_weight=3,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1,
        tree_method='hist', device='cuda',
        verbosity=0
    )
    kf = KFold(n_splits=10, shuffle=True,
               random_state=42)
    qwks, eas, aas, sps = [], [], [], []

    for tr_idx, te_idx in kf.split(X):
        m = XGBRegressor(**PARAMS)
        m.fit(X[tr_idx], y[tr_idx] / 10)
        p = np.clip(
            np.round(m.predict(X[te_idx]) * 10
                     ).astype(int), 0, 10
        )
        yt = y[te_idx].astype(int)
        qwks.append(cohen_kappa_score(
            yt, p, weights='quadratic'))
        eas.append(float(np.mean(yt == p)))
        aas.append(float(np.mean(np.abs(yt-p) <= 1)))
        r, _ = spearmanr(yt, p)
        sps.append(r)

    res = {
        'Model'    : model_label,
        'Params'   : f"{essay_embs.shape[1]}d",
        'QWK'      : round(float(np.mean(qwks)), 3),
        'EA'       : round(float(np.mean(eas)),  3),
        'AA'       : round(float(np.mean(aas)),  3),
        'Spearman' : round(float(np.mean(sps)),  3),
        'QWK_std'  : round(float(np.std(qwks)),  3),
    }
    print(f"\n  ✅ {model_label}")
    print(f"     QWK={res['QWK']:.3f}±{res['QWK_std']:.3f}  "
          f"EA={res['EA']:.3f}  "
          f"AA={res['AA']:.3f}  "
          f"Sp={res['Spearman']:.3f}")
    return res


# ════════════════════════════════════════════════════════════
# RUN MODELS ONE BY ONE
# (GPU memory freed between models automatically)
# ════════════════════════════════════════════════════════════
results = []

# ── MODEL 1: multilingual-e5 (your baseline) ──────────────
print("\n" + "="*55)
print("MODEL 1: multilingual-e5-large-instruct")
print("Type: Decoder-style LLM | 560M params")
print("="*55)
e1, s1, c1 = embed_dataset(
    model_name  = 'intfloat/multilingual-e5-large-instruct',
    essay_instr = (
        "Instruct: Represent this Malayalam student "
        "answer for evaluating content accuracy "
        "against a reference answer\nQuery: "
    ),
    ref_instr   = (
        "Instruct: Represent this Malayalam reference "
        "answer as the gold standard\nQuery: "
    ),
    sent_instr  = (
        "Instruct: Represent this Malayalam sentence "
        "for measuring coherence\nQuery: "
    ),
    checkpoint_name = 'me5',
    use_cls         = False,
    batch_size      = 16,
)
results.append(run_cv(e1, s1, c1, sf_all, y_scores,
                      "multilingual-e5-large-instruct"))
del e1, s1, c1

# ── MODEL 2: LaBSE ────────────────────────────────────────
print("\n" + "="*55)
print("MODEL 2: LaBSE")
print("Type: Encoder | 471M params | 109 languages")
print("="*55)
e2, s2, c2 = embed_dataset(
    model_name      = 'sentence-transformers/LaBSE',
    essay_instr     = "",
    ref_instr       = "",
    sent_instr      = "",
    checkpoint_name = 'labse',
    use_cls         = True,
    batch_size      = 32,
)
results.append(run_cv(e2, s2, c2, sf_all, y_scores,
                      "LaBSE"))
del e2, s2, c2

# ── MODEL 3: MuRIL ────────────────────────────────────────
print("\n" + "="*55)
print("MODEL 3: MuRIL (Google)")
print("Type: Encoder | 236M params | Indian languages")
print("="*55)
e3, s3, c3 = embed_dataset(
    model_name      = 'google/muril-base-cased',
    essay_instr     = "",
    ref_instr       = "",
    sent_instr      = "",
    checkpoint_name = 'muril',
    use_cls         = True,
    batch_size      = 32,
)
results.append(run_cv(e3, s3, c3, sf_all, y_scores,
                      "MuRIL-base"))
del e3, s3, c3

# ── MODEL 4: IndicBERT (ai4bharat) ────────────────────────
print("\n" + "="*55)
print("MODEL 4: IndicBERT (ai4bharat)")
print("Type: Encoder | 112M params | Indic languages")
print("="*55)
e4, s4, c4 = embed_dataset(
    model_name      = 'ai4bharat/indic-bert',
    essay_instr     = "",
    ref_instr       = "",
    sent_instr      = "",
    checkpoint_name = 'indicbert',
    use_cls         = True,
    batch_size      = 32,
)
results.append(run_cv(e4, s4, c4, sf_all, y_scores,
                      "IndicBERT (ai4bharat)"))
del e4, s4, c4

# ── MODEL 5: Qwen2.5-1.5B (decoder LLM) ──────────────────
print("\n" + "="*55)
print("MODEL 5: Qwen2.5-1.5B-Instruct")
print("Type: Decoder LLM | 1.5B params | multilingual")
print("="*55)
e5, s5, c5 = embed_dataset(
    model_name  = 'Qwen/Qwen2.5-1.5B-Instruct',
    essay_instr = (
        "Instruct: Represent this Malayalam student "
        "answer for scoring against a reference\n"
        "Query: "
    ),
    ref_instr   = (
        "Instruct: Represent this Malayalam model "
        "answer as the scoring standard\nQuery: "
    ),
    sent_instr  = (
        "Instruct: Represent this Malayalam sentence "
        "for coherence measurement\nQuery: "
    ),
    checkpoint_name = 'qwen25',
    use_cls         = False,
    batch_size      = 16,
)
results.append(run_cv(e5, s5, c5, sf_all, y_scores,
                      "Qwen2.5-1.5B-Instruct"))
del e5, s5, c5

# ── FINAL TABLE ────────────────────────────────────────────
results_df = pd.DataFrame(results).sort_values(
    'QWK', ascending=False
).reset_index(drop=True)

results_df.to_csv(
    os.path.join(SAVE_DIR, 'mal_model_comparison.csv'),
    index=False
)

print("\n" + "="*65)
print("   MALAYALAM ASAG — MODEL COMPARISON RESULTS")
print("="*65)
print(f"{'#':<3} {'Model':<38} {'Dim':>5} "
      f"{'QWK':>6} {'EA':>6} {'AA':>6} "
      f"{'Spe':>6} {'±':>6}")
print("─"*65)
for i, r in results_df.iterrows():
    print(f"  {i+1:<2} {r['Model']:<38} "
          f"{r['Params']:>5} "
          f"{r['QWK']:>6.3f} "
          f"{r['EA']:>6.3f} "
          f"{r['AA']:>6.3f} "
          f"{r['Spearman']:>6.3f} "
          f"{r['QWK_std']:>6.3f}")
print("─"*65)
print(f"\n  Best: {results_df.iloc[0]['Model']} "
      f"(QWK={results_df.iloc[0]['QWK']:.3f})")
print("\n✅ Model comparison complete!")

✅ Dataset loaded: (3000, 7)
   Columns: ['essay_id', 'prompt_id', 'essay', 'ref_answer', 'essay_score', 'prompt', 'max_marks']
   ref_answer nulls: 0
   Essays: 3000, Prompts: 150

✅ Ready: 3000 essays with reference answers
   Sample ref[0][:80]: സ്വാതന്ത്ര്യത്തിനായുള്ള ഭാരതീയരുടെ പോരാട്ടത്തിന്റെ ഗതി മാറ്റിയ ദിനമാണ് ക്വിറ്റ് ...

Computing V2 shallow features...


100%|██████████| 3000/3000 [00:00<00:00, 3431.22it/s]


✅ Shallow features: (3000, 13)

MODEL 1: multilingual-e5-large-instruct
Type: Decoder-style LLM | 560M params
  ✅ me5 exists — loading...

  ✅ multilingual-e5-large-instruct
     QWK=0.943±0.008  EA=0.581  AA=0.923  Sp=0.946

MODEL 2: LaBSE
Type: Encoder | 471M params | 109 languages
  ✅ labse exists — loading...

  ✅ LaBSE
     QWK=0.941±0.008  EA=0.587  AA=0.919  Sp=0.945

MODEL 3: MuRIL (Google)
Type: Encoder | 236M params | Indian languages
  ✅ muril exists — loading...

  ✅ MuRIL-base
     QWK=0.938±0.008  EA=0.574  AA=0.910  Sp=0.941

MODEL 4: IndicBERT (ai4bharat)
Type: Encoder | 112M params | Indic languages
  ✅ indicbert exists — loading...

  ✅ IndicBERT (ai4bharat)
     QWK=0.936±0.010  EA=0.560  AA=0.904  Sp=0.939

MODEL 5: Qwen2.5-1.5B-Instruct
Type: Decoder LLM | 1.5B params | multilingual
  ✅ qwen25 exists — loading...

  ✅ Qwen2.5-1.5B-Instruct
     QWK=0.928±0.009  EA=0.536  AA=0.885  Sp=0.931

   MALAYALAM ASAG — MODEL COMPARISON RESULTS
#   Model                     

In [19]:
# ============================================================
# Per-Prompt Analysis
# Which prompts are hardest to grade?
# ============================================================

import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')

SAVE_DIR = '/kaggle/working/'

# Load dataset
df   = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/mal_dataset.csv'
)
q_df = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/questions_150.csv'
)
df   = df.merge(
    q_df[['prompt_id','prompt','max_marks']],
    on='prompt_id', how='left'
)

# Load best embeddings (multilingual-e5)
emb_data = np.load(
    os.path.join(SAVE_DIR, 'mal_emb_me5.npz')
)
essay_embs = emb_data['essay_embs']
er_sims    = emb_data['er_sims']
coherence  = emb_data['coherence']

# Rebuild V2 shallow features
def get_sf_v2(essay, ref):
    essay = str(essay); ref = str(ref)
    words = essay.split(); n_w = max(len(words), 1)
    sents = [s.strip() for s in
             re.split(r'[.?!\u0964]', essay)
             if s.strip()]
    n_s   = max(len(sents), 1)
    paras = [p.strip() for p in essay.split('\n')
             if p.strip()]
    ref_w = max(len(ref.split()), 1)
    ref_s = max(len([s for s in
              re.split(r'[.?!\u0964]', ref)
              if s.strip()]), 1)
    mal   = sum(1 for c in essay
                if '\u0D00' <= c <= '\u0D7F')
    alp   = max(sum(1 for c in essay if c.isalnum()), 1)
    sl    = [len(s.split()) for s in sents]
    return [
        n_w, n_s, max(len(paras),1), len(essay),
        n_w/ref_w, n_s/ref_s, ref_w,
        n_w/n_s,
        float(np.std(sl)) if len(sl)>1 else 0.0,
        float(np.mean([len(w) for w in words]))
             if words else 0.0,
        len(set(words))/n_w, mal/alp,
        len(set(c for c in essay
                if not c.isalnum()
                and not c.isspace()))
    ]

from tqdm import tqdm
sf_all = np.array([
    get_sf_v2(r['essay'], r['ref_answer'])
    for _, r in tqdm(df.iterrows(), total=len(df))
], dtype=np.float32)

# Full feature matrix
X = np.nan_to_num(
    np.hstack([
        sf_all, coherence,
        er_sims.reshape(-1,1), essay_embs
    ]).astype(np.float32),
    nan=0.0, posinf=0.0, neginf=0.0
)
y           = df['essay_score'].values.astype(np.float32)
prompt_ids  = df['prompt_id'].values
prompt_text = df.groupby('prompt_id')['prompt'].first()

PARAMS = dict(
    n_estimators=800, learning_rate=0.03,
    max_depth=6, subsample=0.8,
    colsample_bytree=0.7, min_child_weight=3,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1,
    tree_method='hist', device='cuda', verbosity=0
)

# ── Get OOF predictions ────────────────────────────────────
print("Computing OOF predictions...")
kf       = KFold(n_splits=10, shuffle=True,
                 random_state=42)
oof_pred = np.zeros(len(X))

for tr_idx, te_idx in kf.split(X):
    m = XGBRegressor(**PARAMS)
    m.fit(X[tr_idx], y[tr_idx]/10)
    oof_pred[te_idx] = np.clip(
        np.round(m.predict(X[te_idx])*10
                 ).astype(int), 0, 10
    )

# ── Per-prompt stats ───────────────────────────────────────
records = []
for pid in sorted(np.unique(prompt_ids)):
    idx   = np.where(prompt_ids == pid)[0]
    yt    = y[idx].astype(int)
    yp    = oof_pred[idx].astype(int)
    n_uniq = len(np.unique(yt))
    if n_uniq < 2:
        qwk_v = np.nan
    else:
        qwk_v = cohen_kappa_score(
            yt, yp, weights='quadratic'
        )
    records.append({
        'prompt_id'  : int(pid),
        'prompt_text': str(
            prompt_text.get(pid,''))[:60],
        'n_essays'   : len(idx),
        'mean_score' : round(float(np.mean(yt)), 2),
        'score_std'  : round(float(np.std(yt)),  2),
        'qwk'        : round(float(qwk_v), 3)
                        if not np.isnan(qwk_v) else None,
        'ea'         : round(float(
                        np.mean(yt==yp)), 3),
        'aa'         : round(float(
                        np.mean(np.abs(yt-yp)<=1)), 3),
        'mean_error' : round(float(
                        np.mean(np.abs(yt-yp))), 3),
    })

pp_df = pd.DataFrame(records).dropna(subset=['qwk'])
pp_df.to_csv(
    os.path.join(SAVE_DIR,
                 'mal_per_prompt_analysis.csv'),
    index=False
)

# ── Summary stats ──────────────────────────────────────────
print("\n" + "="*60)
print("  PER-PROMPT ANALYSIS — Summary")
print("="*60)
print(f"  Prompts analysed:   {len(pp_df)}")
print(f"  Mean QWK:           {pp_df['qwk'].mean():.3f}")
print(f"  Std  QWK:           {pp_df['qwk'].std():.3f}")
print(f"  Min  QWK (hardest): {pp_df['qwk'].min():.3f} "
      f"→ prompt {pp_df.loc[pp_df['qwk'].idxmin(),'prompt_id']}")
print(f"  Max  QWK (easiest): {pp_df['qwk'].max():.3f} "
      f"→ prompt {pp_df.loc[pp_df['qwk'].idxmax(),'prompt_id']}")

# Hardest and easiest prompts
print("\n  5 HARDEST prompts (lowest QWK):")
hard = pp_df.nsmallest(5, 'qwk')
for _, r in hard.iterrows():
    print(f"  Prompt {r['prompt_id']:>3}: "
          f"QWK={r['qwk']:.3f}  "
          f"score_std={r['score_std']:.2f}  "
          f"{r['prompt_text'][:45]}...")

print("\n  5 EASIEST prompts (highest QWK):")
easy = pp_df.nlargest(5, 'qwk')
for _, r in easy.iterrows():
    print(f"  Prompt {r['prompt_id']:>3}: "
          f"QWK={r['qwk']:.3f}  "
          f"score_std={r['score_std']:.2f}  "
          f"{r['prompt_text'][:45]}...")

# ── QWK buckets ────────────────────────────────────────────
print("\n  QWK Distribution:")
bins   = [0, 0.7, 0.8, 0.9, 0.95, 1.01]
labels = ['<0.7','0.7–0.8','0.8–0.9',
          '0.9–0.95','≥0.95']
pp_df['qwk_bucket'] = pd.cut(
    pp_df['qwk'], bins=bins, labels=labels,
    right=False
)
bucket_counts = pp_df['qwk_bucket'].value_counts(
).sort_index()
for bucket, count in bucket_counts.items():
    bar = '█' * count
    print(f"    {bucket:<10}: {count:>3} prompts  {bar}")

# ── Visualisations ─────────────────────────────────────────
NAVY  = "#1A2E5A"
TEAL  = "#0D9488"
GREY  = "#F8FAFC"

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.patch.set_facecolor(GREY)
fig.suptitle(
    "Per-Prompt Analysis — Malayalam ASAG V2\n"
    "multilingual-e5-large-instruct + XGBoost",
    fontsize=13, fontweight='bold', color=NAVY
)

# Plot 1: QWK distribution across prompts
ax = axes[0, 0]
ax.set_facecolor(GREY)
ax.hist(pp_df['qwk'], bins=20,
        color=TEAL, edgecolor='white', alpha=0.9)
ax.axvline(pp_df['qwk'].mean(), color=NAVY, lw=2,
           linestyle='--',
           label=f"Mean = {pp_df['qwk'].mean():.3f}")
ax.axvline(pp_df['qwk'].median(), color='#F97316',
           lw=2, linestyle=':',
           label=f"Median = {pp_df['qwk'].median():.3f}")
ax.set_xlabel("QWK per Prompt", fontsize=11)
ax.set_ylabel("Number of Prompts", fontsize=11)
ax.set_title("Distribution of Per-Prompt QWK",
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

# Plot 2: Score std vs QWK scatter
ax = axes[0, 1]
ax.set_facecolor(GREY)
sc = ax.scatter(
    pp_df['score_std'], pp_df['qwk'],
    c=pp_df['mean_score'],
    cmap='RdYlGn', alpha=0.7,
    s=55, edgecolors='white', linewidths=0.5
)
plt.colorbar(sc, ax=ax, label='Mean Score')
ax.set_xlabel("Score Std Dev per Prompt", fontsize=11)
ax.set_ylabel("QWK", fontsize=11)
ax.set_title("Score Variance vs Grading Accuracy\n"
             "Higher variance → easier to differentiate",
             fontsize=11, fontweight='bold')
ax.spines[['top','right']].set_visible(False)

# Plot 3: Mean error per prompt (sorted)
ax = axes[1, 0]
ax.set_facecolor(GREY)
sorted_err = pp_df.sort_values('mean_error')
colors     = [TEAL if e <= 0.5 else
              '#F97316' if e <= 1.0 else
              '#E53E3E'
              for e in sorted_err['mean_error']]
ax.bar(range(len(sorted_err)),
       sorted_err['mean_error'],
       color=colors, edgecolor='none', width=1.0)
ax.axhline(sorted_err['mean_error'].mean(),
           color=NAVY, lw=2, linestyle='--',
           label=f"Mean = {sorted_err['mean_error'].mean():.3f}")
ax.set_xlabel("Prompts (sorted by error)", fontsize=11)
ax.set_ylabel("Mean Absolute Error", fontsize=11)
ax.set_title("Per-Prompt Mean Absolute Error\n"
             "Green ≤0.5 | Orange ≤1.0 | Red >1.0",
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

# Plot 4: Mean score vs QWK
ax = axes[1, 1]
ax.set_facecolor(GREY)
ax.scatter(
    pp_df['mean_score'], pp_df['qwk'],
    color=TEAL, alpha=0.6,
    s=50, edgecolors='white'
)
# Trend line
z   = np.polyfit(pp_df['mean_score'], pp_df['qwk'], 1)
xfit = np.linspace(pp_df['mean_score'].min(),
                   pp_df['mean_score'].max(), 100)
ax.plot(xfit, np.polyval(z, xfit),
        color=NAVY, lw=2, linestyle='--',
        label='Trend')
ax.set_xlabel("Mean Score per Prompt", fontsize=11)
ax.set_ylabel("QWK per Prompt", fontsize=11)
ax.set_title("Mean Score vs Grading Accuracy",
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(
    os.path.join(SAVE_DIR,
                 'viz_per_prompt_analysis.png'),
    dpi=150, bbox_inches='tight', facecolor=GREY
)
plt.close()
print("\n✅ viz_per_prompt_analysis.png saved!")
print("✅ mal_per_prompt_analysis.csv saved!")

100%|██████████| 3000/3000 [00:01<00:00, 2617.13it/s]


Computing OOF predictions...

  PER-PROMPT ANALYSIS — Summary
  Prompts analysed:   150
  Mean QWK:           0.937
  Std  QWK:           0.053
  Min  QWK (hardest): 0.726 → prompt 109
  Max  QWK (easiest): 1.000 → prompt 16

  5 HARDEST prompts (lowest QWK):
  Prompt 109: QWK=0.726  score_std=2.80  റിട്ടുകൾ എന്നാൽ എന്ത്? അവയുടെ പ്രാധാന്യം വിലയ...
  Prompt 142: QWK=0.732  score_std=2.57  ഏഷ്യ വൻകരകയുടെ സവിശേഷതകൾ വ്യക്തമാക്കുക....
  Prompt 108: QWK=0.777  score_std=2.48  9,10 പഞ്ചവത്സര പദ്ധതികളുടെ പ്രത്യേകതകൾ എന്തെല...
  Prompt 118: QWK=0.785  score_std=2.40  ചൊവ്വ ഗ്രഹത്തിന്റെ 10 സവിശേഷതകൾ എഴുതുക....
  Prompt  56: QWK=0.798  score_std=2.42  ശിലായുഗ കാലഘട്ടത്തെ നാലായി തരം തിരിച്ചിട്ടുണ്...

  5 EASIEST prompts (highest QWK):
  Prompt  16: QWK=1.000  score_std=2.83  രാഷ്ട്രത്തിൻറെ  ചുമതലകളെ കുറിച്ച് കുറിപ്പ് തയ...
  Prompt  33: QWK=1.000  score_std=2.63  ഇന്ത്യൻ ഗവൺമെൻറിൻറെ ഘടനയെ കുറിച്ച് വിശദമാക്കു...
  Prompt  55: QWK=1.000  score_std=2.83  ഭൂപടങ്ങളിലെ പ്രധാന ഘടകങ്ങളെ കുറിച്ച് വിശദമാക്

In [20]:
# ============================================================
#Feature Ablation Study
# Removes each feature group and measures QWK drop
# Validates contribution of word_ratio and other groups
# ============================================================

print("Running feature ablation study...")

def run_quick_cv(X_feat, y, label):
    kf   = KFold(n_splits=10, shuffle=True,
                 random_state=42)
    qwks = []
    for tr, te in kf.split(X_feat):
        m = XGBRegressor(**PARAMS)
        m.fit(X_feat[tr], y[tr]/10)
        p = np.clip(
            np.round(m.predict(X_feat[te])*10
                     ).astype(int), 0, 10)
        qwks.append(cohen_kappa_score(
            y[te], p, weights='quadratic'))
    print(f"  {label:<45} QWK={np.mean(qwks):.4f}")
    return round(np.mean(qwks), 4)

# Define index ranges in feature vector
N_SF   = len(SF_COLS_V2)          # 0:13
N_COH  = 3                         # 13:16
N_SIM  = 1                         # 16:17
N_EMB  = EMB_DIM                   # 17:17+EMB_DIM

ablation = {}

# Full model
ablation['Full model'] = run_quick_cv(
    X, y, "Full  model (all features)"
)

# Remove entire shallow features
mask  = list(range(N_SF, X.shape[1]))
ablation['Without shallow features'] = run_quick_cv(
    X[:, mask], y, "Without shallow features"
)

# Remove only word_ratio (column index 4 in SF)
w_ratio_idx = SF_COLS_V2.index('word_ratio')
mask2 = [i for i in range(X.shape[1])
         if i != w_ratio_idx]
ablation['Without word_ratio only'] = run_quick_cv(
    X[:, mask2], y, "Without word_ratio "
)

# Remove coherence features
mask3 = (list(range(N_SF)) +
         list(range(N_SF+N_COH, X.shape[1])))
ablation['Without coherence features'] = run_quick_cv(
    X[:, mask3], y, "Without coherence features"
)

# Remove essay_ref_similarity
mask4 = (list(range(N_SF+N_COH)) +
         list(range(N_SF+N_COH+N_SIM, X.shape[1])))
ablation['Without essay_ref_similarity'] = run_quick_cv(
    X[:, mask4], y, "Without essay_ref_similarity"
)

# Remove embeddings (only shallow features)
mask5 = list(range(N_SF+N_COH+N_SIM))
ablation['Without embeddings (SF only)'] = run_quick_cv(
    X[:, mask5], y, "Without embeddings (SF+coh+sim only)"
)

# Embeddings only (no shallow features)
mask6 = list(range(N_SF, X.shape[1]))
ablation['Embeddings only (no SF)'] = run_quick_cv(
    X[:, mask6], y, "Embeddings only (no shallow)"
)

# Save ablation
abl_df = pd.DataFrame([
    {'Configuration': k,
     'QWK': v,
     'Drop vs Full': round(ablation['Full model']-v, 4)}
    for k, v in ablation.items()
]).sort_values('QWK', ascending=False)

abl_df.to_csv(
    os.path.join(SAVE_DIR, 'mal_ablation.csv'),
    index=False
)

print("\n" + "="*60)
print("   ABLATION STUDY RESULTS")
print("="*60)
print(abl_df.to_string(index=False))
print("\n✅ Ablation study complete!")

# Visualise ablation
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor("#F8FAFC")
ax.set_facecolor("#F8FAFC")

configs = abl_df['Configuration'].tolist()
qwks    = abl_df['QWK'].tolist()
colors  = ["#0D9488" if q == max(qwks)
           else "#1D4ED8" if q == min(qwks)
           else "#1A2E5A" for q in qwks]

bars = ax.barh(configs, qwks, color=colors,
               edgecolor='white', height=0.6)
for bar, q in zip(bars, qwks):
    ax.text(q + 0.001,
            bar.get_y()+bar.get_height()/2,
            f"{q:.4f}", va='center',
            fontsize=10, color="#1A2E5A",
            fontweight='bold')

ax.set_xlabel("QWK Score", fontsize=12)
ax.set_title("Ablation Study — Feature Group Contributions\n"
             "Malayalam ASAG ",
             fontsize=12, fontweight='bold', color="#1A2E5A")
ax.set_xlim(0.7, 1.0)
ax.axvline(ablation['Full model'],
           color="#0D9488", lw=2,
           linestyle='--', alpha=0.5,
           label="Full model QWK")
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(
    os.path.join(SAVE_DIR, 'viz_ablation.png'),
    dpi=150, bbox_inches='tight', facecolor="#F8FAFC"
)
plt.close()
print("✅ viz_ablation_.png saved!")

Running feature ablation study...
  Full  model (all features)                    QWK=0.9433
  Without shallow features                      QWK=0.8039
  Without word_ratio                            QWK=0.9416
  Without coherence features                    QWK=0.9431
  Without essay_ref_similarity                  QWK=0.9400
  Without embeddings (SF+coh+sim only)          QWK=0.9329
  Embeddings only (no shallow)                  QWK=0.8039

   ABLATION STUDY RESULTS
               Configuration    QWK  Drop vs Full
                  Full model 0.9433        0.0000
  Without coherence features 0.9431        0.0002
     Without word_ratio only 0.9416        0.0017
Without essay_ref_similarity 0.9400        0.0033
Without embeddings (SF only) 0.9329        0.0104
    Without shallow features 0.8039        0.1394
     Embeddings only (no SF) 0.8039        0.1394

✅ Ablation study complete!
✅ viz_ablation_.png saved!


In [40]:
# ============================================================
# CROSS-PROMPT GENERALISATION 
# ============================================================

import os, re, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import cohen_kappa_score
from xgboost import XGBRegressor
from scipy.stats import spearmanr
from tqdm import tqdm
warnings.filterwarnings('ignore')

SAVE_DIR = '/kaggle/working/'

# ── Load dataset ───────────────────────────────────────────
df   = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/mal_dataset.csv'
)
q_df = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/questions_150.csv'
)
df   = df.merge(
    q_df[['prompt_id','prompt','max_marks']],
    on='prompt_id', how='left'
)
print(f"Dataset: {df.shape}")

# ── Load embeddings ────────────────────────────────────────
d          = np.load(
    os.path.join(SAVE_DIR, 'mal_emb_me5.npz')
)
essay_embs = d['essay_embs'].astype(np.float32)
er_sims    = d['er_sims'].astype(np.float32)
coherence  = d['coherence'].astype(np.float32)
print(f"Embeddings: {essay_embs.shape}")

# ── V2 shallow features ────────────────────────────────────
def sf_v2(essay, ref):
    e=str(essay); r=str(ref)
    ws=e.split(); n_w=max(len(ws),1)
    ss=[s.strip() for s in
        re.split(r'[.?!\u0964]',e) if s.strip()]
    n_s=max(len(ss),1)
    ps=[p.strip() for p in e.split('\n') if p.strip()]
    rw=max(len(r.split()),1)
    rs=max(len([s for s in
        re.split(r'[.?!\u0964]',r) if s.strip()]),1)
    mal=sum(1 for c in e if '\u0D00'<=c<='\u0D7F')
    alp=max(sum(1 for c in e if c.isalnum()),1)
    sl=[len(s.split()) for s in ss]
    return [
        n_w,n_s,max(len(ps),1),len(e),
        n_w/rw,n_s/rs,rw,n_w/n_s,
        float(np.std(sl)) if len(sl)>1 else 0.,
        float(np.mean([len(w) for w in ws]))
        if ws else 0.,
        len(set(ws))/n_w,mal/alp,
        len(set(c for c in e
            if not c.isalnum() and not c.isspace()))
    ]

print("Computing shallow features...")
sf_all = np.array([
    sf_v2(r['essay'], r['ref_answer'])
    for _, r in tqdm(df.iterrows(), total=len(df))
], dtype=np.float32)

# ── Build full feature matrix ──────────────────────────────
X = np.nan_to_num(
    np.hstack([
        sf_all,
        coherence,
        er_sims.reshape(-1, 1),
        essay_embs
    ]).astype(np.float32),
    nan=0., posinf=0., neginf=0.
)
y      = df['essay_score'].values.astype(np.float32)
groups = df['prompt_id'].values

print(f"Feature matrix: {X.shape}")
print(f"Unique prompts: {len(np.unique(groups))}")

# ── XGBoost params ─────────────────────────────────────────
PARAMS = dict(
    n_estimators=800, learning_rate=0.03,
    max_depth=6, subsample=0.8,
    colsample_bytree=0.7, min_child_weight=3,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1,
    tree_method='hist', device='cuda',
    verbosity=0
)

# ── Cross-prompt GroupKFold ────────────────────────────────
print("\n" + "="*55)
print("  CROSS-PROMPT GENERALISATION")
print("  Train: 120 prompts → Test: 30 unseen prompts")
print("="*55)

gkf  = GroupKFold(n_splits=5)
qwks, eas, aas, sps = [], [], [], []

for fold_num, (tr_idx, te_idx) in enumerate(
    gkf.split(X, y, groups), 1
):
    tr_prompts = set(groups[tr_idx])
    te_prompts = set(groups[te_idx])

    print(f"\n  Fold {fold_num}:")
    print(f"    Train: {len(tr_prompts)} prompts, "
          f"{len(tr_idx)} essays")
    print(f"    Test:  {len(te_prompts)} prompts, "
          f"{len(te_idx)} essays  "
          f"(completely unseen)")

    m = XGBRegressor(**PARAMS)
    m.fit(X[tr_idx], y[tr_idx]/10)

    p  = np.clip(
        np.round(
            m.predict(X[te_idx]) * 10
        ).astype(int), 0, 10
    )
    yt = y[te_idx].astype(int)

    qwk_v = cohen_kappa_score(
        yt, p, weights='quadratic'
    )
    ea_v  = float(np.mean(yt == p))
    aa_v  = float(np.mean(np.abs(yt - p) <= 1))
    r, _  = spearmanr(yt, p)

    qwks.append(qwk_v)
    eas.append(ea_v)
    aas.append(aa_v)
    sps.append(r)

    print(f"    QWK = {qwk_v:.3f}  "
          f"EA = {ea_v:.3f}  "
          f"AA = {aa_v:.3f}  "
          f"Sp = {r:.3f}")

# ── Final results ──────────────────────────────────────────
print(f"\n{'='*55}")
print("  FINAL RESULTS")
print(f"{'='*55}")
print(f"  QWK:      "
      f"{np.mean(qwks):.3f} ± {np.std(qwks):.3f}")
print(f"  EA:       {np.mean(eas):.3f}")
print(f"  AA:       {np.mean(aas):.3f}")
print(f"  Spearman: {np.mean(sps):.3f}")

std_qwk   = 0.943
cross_qwk = np.mean(qwks)
gap       = std_qwk - cross_qwk

print(f"\n  Standard 10-fold CV QWK  : {std_qwk:.3f}")
print(f"  Cross-prompt QWK         : {cross_qwk:.3f}")
print(f"  Generalisation gap       : {gap:.3f}")

if gap < 0.02:
    verdict = "✅ Excellent generalisation"
elif gap < 0.05:
    verdict = "🔶 Good generalisation (small drop)"
elif gap < 0.10:
    verdict = "⚠️  Moderate generalisation gap"
else:
    verdict = "❌ Large generalisation gap"
print(f"  Verdict                  : {verdict}")

# Save results
pd.DataFrame([{
    'QWK'          : round(np.mean(qwks), 3),
    'QWK_std'      : round(np.std(qwks),  3),
    'EA'           : round(np.mean(eas),   3),
    'AA'           : round(np.mean(aas),   3),
    'Spearman'     : round(np.mean(sps),   3),
    'std_cv_qwk'   : std_qwk,
    'gap'          : round(gap, 3),
}]).to_csv(
    os.path.join(SAVE_DIR,
                 'mal_cross_prompt_results.csv'),
    index=False
)
print("\n✅ Cross-prompt analysis complete!")

Dataset: (3000, 7)
Embeddings: (3000, 1024)
Computing shallow features...


100%|██████████| 3000/3000 [00:01<00:00, 2564.14it/s]


Feature matrix: (3000, 1041)
Unique prompts: 150

  CROSS-PROMPT GENERALISATION
  Train: 120 prompts → Test: 30 unseen prompts

  Fold 1:
    Train: 120 prompts, 2400 essays
    Test:  30 prompts, 600 essays  (completely unseen)
    QWK = 0.940  EA = 0.553  AA = 0.907  Sp = 0.942

  Fold 2:
    Train: 120 prompts, 2400 essays
    Test:  30 prompts, 600 essays  (completely unseen)
    QWK = 0.930  EA = 0.537  AA = 0.880  Sp = 0.931

  Fold 3:
    Train: 120 prompts, 2400 essays
    Test:  30 prompts, 600 essays  (completely unseen)
    QWK = 0.933  EA = 0.567  AA = 0.903  Sp = 0.934

  Fold 4:
    Train: 120 prompts, 2400 essays
    Test:  30 prompts, 600 essays  (completely unseen)
    QWK = 0.926  EA = 0.552  AA = 0.895  Sp = 0.924

  Fold 5:
    Train: 120 prompts, 2400 essays
    Test:  30 prompts, 600 essays  (completely unseen)
    QWK = 0.935  EA = 0.557  AA = 0.890  Sp = 0.945

  FINAL RESULTS
  QWK:      0.933 ± 0.005
  EA:       0.553
  AA:       0.895
  Spearman: 0.935

  Sta

In [50]:
# ============================================================
# CELL 1 — Install 
# ============================================================
!pip install deep-translator -q

In [51]:
# ============================================================
# CELL 2 — COMPLETE MALAYALAM ASAG: GRADE + FEEDBACK
# ============================================================

import os, re, gc, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import (AutoTokenizer, AutoModel,
                          AutoModelForCausalLM)
from sklearn.metrics.pairwise import cosine_similarity
from xgboost import XGBRegressor
from deep_translator import GoogleTranslator
from kaggle_secrets import UserSecretsClient
from tqdm import tqdm
warnings.filterwarnings('ignore')

SAVE_DIR = '/kaggle/working/'
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

# ════════════════════════════════════════════════════════════
# STEP 1 — Load dataset
# ════════════════════════════════════════════════════════════
print("Step 1: Loading dataset...")
df   = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/mal_dataset.csv'
)
q_df = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/questions_150.csv'
)
df   = df.merge(
    q_df[['prompt_id','prompt','max_marks']],
    on='prompt_id', how='left'
)
print(f"✅ Dataset: {df.shape}")

# ════════════════════════════════════════════════════════════
# STEP 2 — V2 shallow features
# ════════════════════════════════════════════════════════════
def sf_v2(essay, ref):
    e  = str(essay); r = str(ref)
    ws = e.split(); n_w = max(len(ws), 1)
    ss = [s.strip() for s in
          re.split(r'[.?!\u0964]', e) if s.strip()]
    n_s = max(len(ss), 1)
    ps  = [p.strip() for p in
           e.split('\n') if p.strip()]
    rw  = max(len(r.split()), 1)
    rs  = max(len([s for s in
           re.split(r'[.?!\u0964]', r)
           if s.strip()]), 1)
    mal = sum(1 for c in e
              if '\u0D00' <= c <= '\u0D7F')
    alp = max(sum(1 for c in e if c.isalnum()), 1)
    sl  = [len(s.split()) for s in ss]
    return [
        n_w, n_s, max(len(ps), 1), len(e),
        n_w/rw, n_s/rs, rw, n_w/n_s,
        float(np.std(sl)) if len(sl)>1 else 0.,
        float(np.mean([len(w) for w in ws]))
             if ws else 0.,
        len(set(ws))/n_w, mal/alp,
        len(set(c for c in e
                if not c.isalnum()
                and not c.isspace()))
    ]

print("Step 2: Computing features...")
sf_all = np.array([
    sf_v2(r['essay'], r['ref_answer'])
    for _, r in tqdm(df.iterrows(),
                     total=len(df))
], dtype=np.float32)
print(f"✅ Shallow features: {sf_all.shape}")

# ════════════════════════════════════════════════════════════
# STEP 3 — Load embeddings and train XGBoost
# ════════════════════════════════════════════════════════════
print("\nStep 3: Loading embeddings...")
d  = np.load(os.path.join(SAVE_DIR, 'mal_emb_me5.npz'))
X  = np.nan_to_num(np.hstack([
    sf_all,
    d['coherence'].astype(np.float32),
    d['er_sims'].reshape(-1,1).astype(np.float32),
    d['essay_embs'].astype(np.float32)
]).astype(np.float32),
    nan=0., posinf=0., neginf=0.)
y  = df['essay_score'].values.astype(np.float32)

print("Training XGBoost on full data...")
xgb = XGBRegressor(
    n_estimators=800, learning_rate=0.03,
    max_depth=6, subsample=0.8,
    colsample_bytree=0.7, min_child_weight=3,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1,
    tree_method='hist', device='cuda',
    verbosity=0
)
xgb.fit(X, y/10)
print("✅ XGBoost ready!")

# ════════════════════════════════════════════════════════════
# STEP 4 — Load scoring model (multilingual-e5)
# ════════════════════════════════════════════════════════════
print("\nStep 4: Loading scoring model...")
gc.collect(); torch.cuda.empty_cache()

EMB_MODEL = 'intfloat/multilingual-e5-large-instruct'
emb_tok   = AutoTokenizer.from_pretrained(
    EMB_MODEL, token=HF_TOKEN
)
emb_mdl   = AutoModel.from_pretrained(
    EMB_MODEL, dtype=torch.float16,
    token=HF_TOKEN
).to('cuda')
emb_mdl.eval()
print(f"✅ Scoring model ready! "
      f"GPU:{torch.cuda.mem_get_info()[0]/1024**3:.1f}GB")

# ════════════════════════════════════════════════════════════
# STEP 5 — Load Gemma for English feedback
# ════════════════════════════════════════════════════════════
print("\nStep 5: Loading Gemma-2-2b-it...")
FB_MODEL = 'google/gemma-2-2b-it'
fb_tok   = AutoTokenizer.from_pretrained(
    FB_MODEL, token=HF_TOKEN
)
fb_mdl   = AutoModelForCausalLM.from_pretrained(
    FB_MODEL, dtype=torch.float16,
    token=HF_TOKEN
).to('cuda')
fb_mdl.eval()
print(f"✅ Gemma ready! "
      f"GPU:{torch.cuda.mem_get_info()[0]/1024**3:.1f}GB")

# ════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ════════════════════════════════════════════════════════════

ESSAY_I = (
    "Instruct: Represent this Malayalam student "
    "answer for evaluating content accuracy "
    "against a reference answer\nQuery: "
)
REF_I = (
    "Instruct: Represent this Malayalam reference "
    "answer as the gold standard\nQuery: "
)
SENT_I = (
    "Instruct: Represent this Malayalam sentence "
    "for measuring coherence\nQuery: "
)

def get_emb(text, instr):
    enc = emb_tok(
        [instr + str(text)[:500]],
        max_length=512, padding=True,
        truncation=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        out = emb_mdl(**enc)
        seq = enc['attention_mask'].sum(dim=1) - 1
        emb = out.last_hidden_state[
            torch.arange(1,
                device=out.last_hidden_state.device),
            seq
        ]
        emb = F.normalize(emb.float(), p=2, dim=1)
    return emb.cpu().numpy()[0]

def get_coh(text):
    ss = [s.strip() for s in
          re.split(r'[.?!\u0964]', str(text))
          if s.strip() and len(s.split()) > 2][:15]
    if len(ss) < 2:
        return 0., 0., 0.
    se   = np.array([get_emb(s, SENT_I) for s in ss])
    sims = [
        float(cosine_similarity(
            se[i].reshape(1,-1),
            se[i+1].reshape(1,-1)
        )[0][0])
        for i in range(len(se)-1)
    ]
    return (float(np.mean(sims)),
            float(np.min(sims)),
            float(np.std(sims)))

def find_missing(student, ref):
    ref_parts = [s.strip() for s in
                 re.split(r'[.،,\n]', str(ref))
                 if len(s.strip()) > 10]
    stu_lower = str(student).lower()
    missing   = []
    for part in ref_parts[:8]:
        words = [w for w in part.split()
                 if len(w) > 4 and
                 '\u0D00' <= w[0] <= '\u0D7F']
        if len(words) < 2:
            continue
        found = sum(1 for w in words[:4]
                    if w.lower() in stu_lower)
        if found < len(words[:4]) * 0.4:
            missing.append(
                part[:60] +
                ("..." if len(part) > 60 else "")
            )
    return missing[:3]

def translate_to_malayalam(text):
    """English → Malayalam via Google Translate."""
    if not text or not text.strip():
        return text
    sentences = re.split(r'(?<=[.!?:])\s+',
                         text.strip())
    translated = []
    for sent in sentences:
        if not sent.strip():
            continue
        try:
            result = GoogleTranslator(
                source='en', target='ml'
            ).translate(sent.strip())
            translated.append(result)
            time.sleep(0.2)
        except Exception:
            translated.append(sent)
    return ' '.join(translated)

def gemma_feedback(prompt_text, ref_answer,
                   student_answer, score,
                   max_marks, missing_pts):
    """Generate English feedback — teacher talking to student."""
    missing_str = (
        "; ".join(missing_pts[:2])
        if missing_pts else "None identified"
    )
    prompt = (
        f"<start_of_turn>user\n"
        f"You are a caring teacher giving personal "
        f"feedback directly to your student. "
        f"Speak to the student as 'you'. "
        f"Be encouraging, specific, and helpful.\n\n"
        f"Question: {str(prompt_text)[:120]}\n"
        f"Expected answer: {str(ref_answer)[:180]}\n"
        f"Student wrote: {str(student_answer)[:180]}\n"
        f"Score: {score}/{max_marks}\n"
        f"Missing points: {missing_str}\n\n"
        f"Write feedback in this format "
        f"(speak directly to the student):\n"
        f"STRENGTH: [Tell the student what they did "
        f"well, starting with 'You...']\n"
        f"WEAKNESS: [Tell the student what is missing "
        f"or incomplete, starting with 'However, "
        f"your answer...']\n"
        f"SUGGESTION: [Give one specific action the "
        f"student can take, starting with 'To improve, "
        f"try to...']\n"
        f"<end_of_turn>\n<start_of_turn>model\n"
    )
    enc = fb_tok(
        [prompt], return_tensors='pt',
        max_length=700, truncation=True
    ).to('cuda')
    with torch.no_grad():
        out = fb_mdl.generate(
            **enc, max_new_tokens=180,
            temperature=0.4, do_sample=True,
            pad_token_id=fb_tok.eos_token_id
        )
    return fb_tok.decode(
        out[0][enc['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()
# ════════════════════════════════════════════════════════════
# MAIN FUNCTION — Grade + Feedback
# ════════════════════════════════════════════════════════════
def grade_and_feedback(
    prompt_text,
    ref_answer,
    student_answer,
    max_marks    = 10,
    actual_score = None
):
    print("\n" + "═"*65)
    print("  MALAYALAM ASAG — GRADE + FEEDBACK REPORT")
    print("═"*65)

    # Print inputs
    print(f"\n  📌 QUESTION:")
    print(f"     {str(prompt_text).strip()}")

    print(f"\n  📖 REFERENCE ANSWER:")
    rw = str(ref_answer).split()
    ln = "     "
    for w in rw[:50]:
        if len(ln)+len(w) > 68:
            print(ln); ln = "     "+w+" "
        else:
            ln += w+" "
    if ln.strip(): print(ln)
    if len(rw) > 50: print("     [...]")

    print(f"\n  ✍️  STUDENT ANSWER:")
    sw = str(student_answer).split()
    ln = "     "
    for w in sw:
        if len(ln)+len(w) > 68:
            print(ln); ln = "     "+w+" "
        else:
            ln += w+" "
    if ln.strip(): print(ln)

    # ── Compute features ──────────────────────────────────
    print(f"\n  ⏳ Computing score...")
    feats      = sf_v2(student_answer, ref_answer)
    e_emb      = get_emb(student_answer, ESSAY_I)
    r_emb      = get_emb(ref_answer, REF_I)
    er_sim     = float(cosine_similarity(
        e_emb.reshape(1,-1), r_emb.reshape(1,-1)
    )[0][0])
    cm, cmin, cstd = get_coh(student_answer)
    ttr        = feats[10]
    word_ratio = feats[4]

    feat = np.nan_to_num(
        np.array(
            feats + [cm, cmin, cstd, er_sim] +
            e_emb.tolist(),
            dtype=np.float32
        ).reshape(1,-1),
        nan=0., posinf=0., neginf=0.
    )
    score = int(np.clip(
        np.round(xgb.predict(feat)[0]*10),
        0, max_marks
    ))
    grade = ("A+" if score/max_marks >= 0.90 else
             "A"  if score/max_marks >= 0.80 else
             "B"  if score/max_marks >= 0.60 else
             "C"  if score/max_marks >= 0.40 else "D")

    # ── Print score ────────────────────────────────────────
    print(f"\n{'─'*65}")
    print(f"  🎯 PREDICTED SCORE : {score} / {max_marks}"
          f"  [{grade}]")
    if actual_score is not None:
        diff = score - actual_score
        ok   = "✅" if abs(diff) <= 1 else "⚠️"
        print(f"  📊 ACTUAL SCORE    : "
              f"{actual_score} / {max_marks}  {ok}")

    # ── Dimension analysis ─────────────────────────────────
    def icon(l):
        return ("✅" if l == "Good" else
                "🔶" if l == "Fair" else "❌")

    dims = {
        "Content Accuracy": (
            "Good" if er_sim >= 0.90 else
            "Fair" if er_sim >= 0.75 else "Needs Work",
            f"similarity = {er_sim:.2f}"
        ),
        "Answer Coverage": (
            "Good" if word_ratio >= 0.75 else
            "Fair" if word_ratio >= 0.40 else "Needs Work",
            f"coverage = {word_ratio:.0%}"
        ),
        "Sentence Flow": (
            "Good" if cm >= 0.85 else
            "Fair" if cm >= 0.70 else "Needs Work",
            f"coherence = {cm:.2f}"
        ),
        "Vocabulary": (
            "Good" if ttr >= 0.70 else
            "Fair" if ttr >= 0.50 else "Needs Work",
            f"diversity = {ttr:.2f}"
        ),
    }
    print(f"\n{'─'*65}")
    print("  📊 DIMENSION ANALYSIS:")
    for dn, (dl, dv) in dims.items():
        print(f"     {icon(dl)} {dn:<22} "
              f"{dl:<12} ({dv})")

    missing = find_missing(student_answer, ref_answer)

    # ── Build English feedback lines ───────────────────────
    fb_en = []

    if er_sim >= 0.90:
        fb_en.append(
            "Content is excellent — "
            "key points accurately covered."
        )
    elif er_sim >= 0.75:
        fb_en.append(
            "Content is partially covered — "
            "some key points are missing."
        )
    else:
        fb_en.append(
            "Content is insufficient — "
            "answer does not adequately "
            "address the question."
        )

    if word_ratio >= 0.75:
        fb_en.append(
            "Answer length is appropriate "
            "and comprehensive."
        )
    elif word_ratio >= 0.40:
        fb_en.append(
            f"Answer is short "
            f"({int(word_ratio*100)}% of expected) "
            f"— add more relevant details."
        )
    else:
        fb_en.append(
            f"Answer is too short "
            f"({int(word_ratio*100)}%) "
            f"— write a much more detailed response."
        )

    if cm >= 0.85:
        fb_en.append(
            "Sentences are well-connected "
            "with good logical flow."
        )
    else:
        fb_en.append(
            "Use linking words like 'therefore', "
            "'because', 'also' to connect ideas."
        )

    if missing and score < max_marks * 0.8:
        fb_en.append(
            "Missing key points: " +
            "; ".join([m[:50] for m in missing[:2]])
        )

    # ── Translate feedback to Malayalam ───────────────────
    print(f"\n{'─'*65}")
    print("  💬 FEEDBACK (English):")
    for i, fb in enumerate(fb_en, 1):
        print(f"     {i}. {fb}")

    print(f"\n  💬 ഫീഡ്ബാക്ക് (Malayalam):")
    print("     ⏳ Translating...")
    for i, fb in enumerate(fb_en, 1):
        ml = translate_to_malayalam(fb)
        print(f"     {i}. {ml}")

    # ── Gemma AI feedback ──────────────────────────────────
    print(f"\n{'─'*65}")
    print("  🤖 AI FEEDBACK (Gemma-2-2b-it):")
    print("  ⏳ Generating English feedback...")
    ai_en = gemma_feedback(
        prompt_text, ref_answer,
        student_answer, score, max_marks, missing
    )

    print(f"\n  [English]")
    for line in ai_en.split('\n'):
        if line.strip():
            print(f"     {line}")

    print(f"\n  ⏳ Translating to Malayalam...")
    ai_ml = translate_to_malayalam(ai_en)

    print(f"\n  [മലയാളം]")
    for line in re.split(r'(?<=[.!?:])\s+', ai_ml):
        if line.strip():
            print(f"     {line}")

    print("═"*65)

    return {
        'score'       : score,
        'grade'       : grade,
        'dimensions'  : dims,
        'feedback_en' : fb_en,
        'ai_en'       : ai_en,
        'ai_ml'       : ai_ml,
    }

print("\n✅ All functions ready!")
print("Run the test cell below.")

Step 1: Loading dataset...
✅ Dataset: (3000, 7)
Step 2: Computing features...


100%|██████████| 3000/3000 [00:01<00:00, 2494.44it/s]


✅ Shallow features: (3000, 13)

Step 3: Loading embeddings...
Training XGBoost on full data...
✅ XGBoost ready!

Step 4: Loading scoring model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ Scoring model ready! GPU:5.6GB

Step 5: Loading Gemma-2-2b-it...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

✅ Gemma ready! GPU:2.3GB

✅ All functions ready!
Run the test cell below.


In [52]:
# ============================================================
# CELL 3 — Run tests
# ============================================================

r1 = grade_and_feedback(
    prompt_text    = "ക്വിറ്റ് ഇന്ത്യ സമരത്തെക്കുറിച്ച് വിശദീകരിക്കുക.",
    ref_answer     = """1942 ഓഗസ്റ്റ് 9-ന് ഗാന്ധിജിയുടെ
    നേതൃത്വത്തിൽ ആരംഭിച്ചു. ചെയ്യുക അല്ലെങ്കിൽ
    മരിക്കുക. ക്രിപ്സ് മിഷൻ പരാജയം.
    ഒരു ലക്ഷം പേർ അറസ്റ്റ്.""",
    student_answer = """ക്വിറ്റ് ഇന്ത്യ 1942-ൽ
    ഗാന്ധിജിയുടെ നേതൃത്വത്തിൽ.
    ചെയ്യുക അല്ലെങ്കിൽ മരിക്കുക.
    ക്രിപ്സ് മിഷൻ പരാജയം.""",
    max_marks      = 10,
    actual_score   = 7
)

r2 = grade_and_feedback(
    prompt_text    = "ഫ്രഞ്ച് വിപ്ലവത്തിന്റെ കാരണങ്ങൾ?",
    ref_answer     = """ഫ്യൂഡൽ വ്യവസ്ഥ, ബൂർബൺ ആഡംബരം,
    ദാരിദ്ര്യം, നികുതി ഭാരം,
    അമേരിക്കൻ സ്വാതന്ത്ര്യ സ്വാധീനം,
    ജ്ഞാനോദയ ആശയങ്ങൾ.""",
    student_answer = "ദാരിദ്ര്യം ഒരു കാരണം.",
    max_marks      = 10,
    actual_score   = 1
)


═════════════════════════════════════════════════════════════════
  MALAYALAM ASAG — GRADE + FEEDBACK REPORT
═════════════════════════════════════════════════════════════════

  📌 QUESTION:
     ക്വിറ്റ് ഇന്ത്യ സമരത്തെക്കുറിച്ച് വിശദീകരിക്കുക.

  📖 REFERENCE ANSWER:
     1942 ഓഗസ്റ്റ് 9-ന് ഗാന്ധിജിയുടെ നേതൃത്വത്തിൽ ആരംഭിച്ചു. ചെയ്യുക 
     അല്ലെങ്കിൽ മരിക്കുക. ക്രിപ്സ് മിഷൻ പരാജയം. ഒരു ലക്ഷം പേർ 
     അറസ്റ്റ്. 

  ✍️  STUDENT ANSWER:
     ക്വിറ്റ് ഇന്ത്യ 1942-ൽ ഗാന്ധിജിയുടെ നേതൃത്വത്തിൽ. ചെയ്യുക 
     അല്ലെങ്കിൽ മരിക്കുക. ക്രിപ്സ് മിഷൻ പരാജയം. 

  ⏳ Computing score...

─────────────────────────────────────────────────────────────────
  🎯 PREDICTED SCORE : 4 / 10  [C]
  📊 ACTUAL SCORE    : 7 / 10  ⚠️

─────────────────────────────────────────────────────────────────
  📊 DIMENSION ANALYSIS:
     ✅ Content Accuracy       Good         (similarity = 0.94)
     🔶 Answer Coverage        Fair         (coverage = 69%)
     ✅ Sentence Flow          Good         (coherence = 0.91)
     ✅ Vocabu